In [1]:
import sys

print(sys.executable)

c:\CricIQ\venv\Scripts\python.exe


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_DIR = Path(r"C:\CricIQ")
DATA_PATH = PROJECT_DIR / "data" / "processed" / "integrated" / "cricketiq_integrated_matches.csv"
FEATURE_DIR = PROJECT_DIR / "data" / "features"

# Create feature output directory if it doesn't exist
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Load integrated match dataset
matches_df = pd.read_csv(DATA_PATH)

# Convert match_date into datetime format
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"])

print("Dataset loaded successfully!")
print("Shape:", matches_df.shape)

print("\nColumns:")
print(matches_df.columns.tolist())

print("\nDate range:")
print(matches_df["match_date"].min(), "to", matches_df["match_date"].max())

print("\nResult distribution:")
print(matches_df["result_type"].value_counts(dropna=False))

print("\nSource distribution:")
print(matches_df["source"].value_counts(dropna=False))

print("\nFirst 5 records:")
display(matches_df.head())


Dataset loaded successfully!
Shape: (2753, 11)

Columns:
['match_date', 'team_1', 'team_2', 'venue', 'city', 'toss_winner', 'toss_decision', 'winner', 'result_type', 'source', 'match_id']

Date range:
2002-06-27 00:00:00 to 2026-10-25 00:00:00

Result distribution:
result_type
Win          2614
No result     111
Tie            28
Name: count, dtype: int64

Source distribution:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

First 5 records:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",Nottingham,England,bat,England,Win,Cricsheet,66283.0
1,2002-06-29,England,India,"Lord's, London",London,England,bat,India,Win,Cricsheet,66284.0
2,2002-12-29,New Zealand,India,"McLean Park, Napier",Napier,India,field,New Zealand,Win,Cricsheet,64814.0
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",Christchurch,India,bat,New Zealand,Win,Cricsheet,64815.0
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",Queenstown,New Zealand,field,New Zealand,Win,Cricsheet,64816.0


In [3]:

# 1. Check missing values
print("Missing values:")
print(matches_df.isnull().sum())

# 2. Check duplicate match records
print("\nDuplicate rows:", matches_df.duplicated().sum())

# 3. Check missing team names
print("\nMissing team names:")
print("Team 1:", matches_df["team_1"].isnull().sum())
print("Team 2:", matches_df["team_2"].isnull().sum())

# 4. Check matches where both teams are identical
same_team_matches = matches_df[
    matches_df["team_1"] == matches_df["team_2"]
]

print("\nMatches with identical team names:", len(same_team_matches))

# 5. Check result categories
print("\nUnique result categories:")
print(matches_df["result_type"].unique())

# 6. Check whether winners belong to either participating team
valid_winner = (
    matches_df["winner"].isna()
    | (matches_df["winner"] == matches_df["team_1"])
    | (matches_df["winner"] == matches_df["team_2"])
)

print("\nInvalid winner records:", (~valid_winner).sum())


Missing values:
match_date         0
team_1             0
team_2             0
venue              0
city               0
toss_winner        4
toss_decision      4
winner           139
result_type        0
source             0
match_id         153
dtype: int64

Duplicate rows: 0

Missing team names:
Team 1: 0
Team 2: 0

Matches with identical team names: 0

Unique result categories:
['Win' 'No result' 'Tie']

Invalid winner records: 0


In [4]:

# Create a copy for feature engineering
features_df = matches_df.copy()

# Create target variable
features_df["team_1_win"] = np.where(
    features_df["winner"] == features_df["team_1"],
    1,
    np.where(
        features_df["winner"] == features_df["team_2"],
        0,
        np.nan
    )
)

# Check target distribution
print("Target distribution:")
print(features_df["team_1_win"].value_counts(dropna=False))

print("\nTarget percentage:")
print(
    features_df["team_1_win"]
    .value_counts(normalize=True, dropna=False) * 100
)

print("\nSample records:")
display(
    features_df[
        ["match_date", "team_1", "team_2", "winner",
         "result_type", "team_1_win"]
    ].head(10)
)


Target distribution:
team_1_win
1.0    1418
0.0    1196
NaN     139
Name: count, dtype: int64

Target percentage:
team_1_win
1.0    51.507446
0.0    43.443516
NaN     5.049037
Name: proportion, dtype: float64

Sample records:


,match_date,team_1,team_2,winner,result_type,team_1_win
0,2002-06-27,England,Sri Lanka,England,Win,1.0
1,2002-06-29,England,India,India,Win,0.0
2,2002-12-29,New Zealand,India,New Zealand,Win,1.0
3,2003-01-01,India,New Zealand,New Zealand,Win,0.0
4,2003-01-04,India,New Zealand,New Zealand,Win,0.0
5,2003-01-08,New Zealand,India,India,Win,0.0
6,2003-01-09,Sri Lanka,Australia,Sri Lanka,Win,1.0
7,2003-01-11,Australia,England,Australia,Win,1.0
8,2003-01-13,Sri Lanka,England,Sri Lanka,Win,1.0
9,2003-01-14,India,New Zealand,New Zealand,Win,0.0


In [5]:

# Sort matches chronologically
features_df = features_df.sort_values(
    by=["match_date"]
).reset_index(drop=True)

# Check chronological ordering
print("Dataset sorted chronologically:",
      features_df["match_date"].is_monotonic_increasing)

# Count matches per date
matches_per_date = features_df.groupby("match_date").size()

print("\nNumber of unique match dates:", len(matches_per_date))
print("Dates with multiple matches:", (matches_per_date > 1).sum())
print("Maximum matches on a single date:", matches_per_date.max())
print()
# Display dates with multiple matches
print("\nSample dates with multiple matches:")
display(matches_per_date[matches_per_date > 1].head(10))

# Display first and last 5 records
print("\nEarliest matches:")
display(features_df.head())

print("\nLatest matches:")
display(features_df.tail())


Dataset sorted chronologically: True

Number of unique match dates: 2326
Dates with multiple matches: 396
Maximum matches on a single date: 4


Sample dates with multiple matches:


match_date
2003-02-10    2
2003-02-11    2
2003-02-12    2
2003-02-15    2
2003-02-16    3
2003-02-19    3
2003-02-22    2
2003-02-23    2
2003-02-24    2
2003-02-26    2
dtype: int64


Earliest matches:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id,team_1_win
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",Nottingham,England,bat,England,Win,Cricsheet,66283.0,1.0
1,2002-06-29,England,India,"Lord's, London",London,England,bat,India,Win,Cricsheet,66284.0,0.0
2,2002-12-29,New Zealand,India,"McLean Park, Napier",Napier,India,field,New Zealand,Win,Cricsheet,64814.0,1.0
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",Christchurch,India,bat,New Zealand,Win,Cricsheet,64815.0,0.0
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",Queenstown,New Zealand,field,New Zealand,Win,Cricsheet,64816.0,0.0



Latest matches:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id,team_1_win
2748,2026-10-22,Ireland,Namibia,"High Performance Oval, Windhoek",Windhoek,Namibia,field,Ireland,Win,Cricsheet,1558231.0,1.0
2749,2026-10-23,Afghanistan,Zimbabwe,Sharjah Cricket Stadium,Sharjah,Zimbabwe,bat,Afghanistan,Win,Crickrida,NaN,1.0
2750,2026-10-24,Namibia,Ireland,"High Performance Oval, Windhoek",Windhoek,Ireland,field,Namibia,Win,Cricsheet,1558232.0,1.0
2751,2026-10-25,England,Pakistan,"Gaddafi Stadium, Lahore",Lahore,Pakistan,field,Pakistan,Win,Cricsheet,1550716.0,0.0
2752,2026-10-25,United Arab Emirates,Netherlands,"ICC Academy, Dubai",Dubai,United Arab Emirates,bat,Netherlands,Win,Cricsheet,1559563.0,0.0


In [6]:

# Store historical wins and decisive matches for each team
team_wins = {}
team_matches = {}

# Create columns for historical performance
features_df["team_1_win_pct"] = np.nan
features_df["team_2_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate historical win percentages BEFORE today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        team1_wins = team_wins.get(team1, 0)
        team1_matches = team_matches.get(team1, 0)

        team2_wins = team_wins.get(team2, 0)
        team2_matches = team_matches.get(team2, 0)

        # Use 0.5 as the neutral starting estimate for teams
        features_df.at[idx, "team_1_win_pct"] = (
            team1_wins / team1_matches
            if team1_matches > 0 else 0.5
        )

        features_df.at[idx, "team_2_win_pct"] = (
            team2_wins / team2_matches
            if team2_matches > 0 else 0.5
        )

    # Update historical statistics AFTER processing all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        # Update statistics only for decisive matches
        if pd.notna(winner):

            team_matches[team1] = team_matches.get(team1, 0) + 1
            team_matches[team2] = team_matches.get(team2, 0) + 1

            if winner == team1:
                team_wins[team1] = team_wins.get(team1, 0) + 1

            elif winner == team2:
                team_wins[team2] = team_wins.get(team2, 0) + 1

# Calculate the difference in historical win percentage
features_df["win_pct_difference"] = (
    features_df["team_1_win_pct"] -
    features_df["team_2_win_pct"]
)

# Inspect results
print("Historical win percentage features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_win_pct", "team_2_win_pct",
         "win_pct_difference", "winner"]
    ].head(15)
)


Historical win percentage features created!


,match_date,team_1,team_2,team_1_win_pct,team_2_win_pct,win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,0.500000,0.500000,0.000000,England
1,2002-06-29,England,India,1.000000,0.500000,0.500000,India
2,2002-12-29,New Zealand,India,0.500000,1.000000,-0.500000,New Zealand
3,2003-01-01,India,New Zealand,0.500000,1.000000,-0.500000,New Zealand
4,2003-01-04,India,New Zealand,0.333333,1.000000,-0.666667,New Zealand
5,2003-01-08,New Zealand,India,1.000000,0.250000,0.750000,India
6,2003-01-09,Sri Lanka,Australia,0.000000,0.500000,-0.500000,Sri Lanka
7,2003-01-11,Australia,England,0.000000,0.500000,-0.500000,Australia
8,2003-01-13,Sri Lanka,England,0.500000,0.333333,0.166667,Sri Lanka
9,2003-01-14,India,New Zealand,0.400000,0.750000,-0.350000,New Zealand


In [7]:

# Store the last 5 decisive results for each team
team_recent_results = {}

# Create recent form columns
features_df["team_1_recent_form"] = np.nan
features_df["team_2_recent_form"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate recent form BEFORE today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        team1_history = team_recent_results.get(team1, [])
        team2_history = team_recent_results.get(team2, [])

        # Neutral 50% if fewer than 5 previous decisive matches exist
        features_df.at[idx, "team_1_recent_form"] = (
            np.mean(team1_history[-5:])
            if team1_history else 0.5
        )

        features_df.at[idx, "team_2_recent_form"] = (
            np.mean(team2_history[-5:])
            if team2_history else 0.5
        )

    # Update recent results only after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):

            team1_result = 1 if winner == team1 else 0
            team2_result = 1 if winner == team2 else 0

            team_recent_results.setdefault(team1, []).append(team1_result)
            team_recent_results.setdefault(team2, []).append(team2_result)

# Calculate recent form difference
features_df["recent_form_difference"] = (
    features_df["team_1_recent_form"] -
    features_df["team_2_recent_form"]
)

# Display sample results
print("Recent form features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_recent_form", "team_2_recent_form",
         "recent_form_difference", "winner"]
    ].head(20)
)
display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_recent_form", "team_2_recent_form",
         "recent_form_difference", "winner"]
    ].tail(20)
)


Recent form features created!


,match_date,team_1,team_2,team_1_recent_form,team_2_recent_form,recent_form_difference,winner
0,2002-06-27,England,Sri Lanka,0.500000,0.500000,0.000000,England
1,2002-06-29,England,India,1.000000,0.500000,0.500000,India
2,2002-12-29,New Zealand,India,0.500000,1.000000,-0.500000,New Zealand
3,2003-01-01,India,New Zealand,0.500000,1.000000,-0.500000,New Zealand
4,2003-01-04,India,New Zealand,0.333333,1.000000,-0.666667,New Zealand
5,2003-01-08,New Zealand,India,1.000000,0.250000,0.750000,India
6,2003-01-09,Sri Lanka,Australia,0.000000,0.500000,-0.500000,Sri Lanka
7,2003-01-11,Australia,England,0.000000,0.500000,-0.500000,Australia
8,2003-01-13,Sri Lanka,England,0.500000,0.333333,0.166667,Sri Lanka
9,2003-01-14,India,New Zealand,0.400000,0.750000,-0.350000,New Zealand


,match_date,team_1,team_2,team_1_recent_form,team_2_recent_form,recent_form_difference,winner
2733,2026-10-09,Oman,Canada,0.2,0.2,0.0,Oman
2734,2026-10-09,United Arab Emirates,Namibia,0.4,0.2,0.2,Namibia
2735,2026-10-11,Nepal,Canada,0.4,0.0,0.4,Nepal
2736,2026-10-11,United Arab Emirates,United States of America,0.4,0.8,-0.4,United States of America
2737,2026-10-13,United States of America,Namibia,0.8,0.2,0.6,Namibia
2738,2026-10-13,Nepal,Oman,0.6,0.2,0.4,Nepal
2739,2026-10-15,Canada,Oman,0.0,0.2,-0.2,Canada
2740,2026-10-17,Canada,Nepal,0.2,0.8,-0.6,Canada
2741,2026-10-17,Zimbabwe,Bangladesh,0.2,0.4,-0.2,Zimbabwe
2742,2026-10-18,Pakistan,Sri Lanka,0.6,0.4,0.2,Pakistan


In [8]:

# Store previous head-to-head results
head_to_head = {}

# Create head-to-head feature columns
features_df["team_1_h2h_win_pct"] = np.nan
features_df["team_2_h2h_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate head-to-head performance before today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        pair = tuple(sorted([team1, team2]))
        previous_results = head_to_head.get(pair, [])

        if previous_results:
            team1_wins = sum(
                1 for winner in previous_results if winner == team1
            )
            team1_pct = team1_wins / len(previous_results)
            team2_pct = 1 - team1_pct
        else:
            team1_pct = 0.5
            team2_pct = 0.5

        features_df.at[idx, "team_1_h2h_win_pct"] = team1_pct
        features_df.at[idx, "team_2_h2h_win_pct"] = team2_pct

    # Update head-to-head history after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):
            pair = tuple(sorted([team1, team2]))
            head_to_head.setdefault(pair, []).append(winner)

# Calculate head-to-head difference
features_df["h2h_win_pct_difference"] = (
    features_df["team_1_h2h_win_pct"] -
    features_df["team_2_h2h_win_pct"]
)

print("Head-to-head features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_h2h_win_pct", "team_2_h2h_win_pct",
         "h2h_win_pct_difference", "winner"]
    ].head(20)
)


Head-to-head features created!


,match_date,team_1,team_2,team_1_h2h_win_pct,team_2_h2h_win_pct,h2h_win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,0.50,0.50,0.0,England
1,2002-06-29,England,India,0.50,0.50,0.0,India
2,2002-12-29,New Zealand,India,0.50,0.50,0.0,New Zealand
3,2003-01-01,India,New Zealand,0.00,1.00,-1.0,New Zealand
4,2003-01-04,India,New Zealand,0.00,1.00,-1.0,New Zealand
5,2003-01-08,New Zealand,India,1.00,0.00,1.0,India
6,2003-01-09,Sri Lanka,Australia,0.50,0.50,0.0,Sri Lanka
7,2003-01-11,Australia,England,0.50,0.50,0.0,Australia
8,2003-01-13,Sri Lanka,England,0.00,1.00,-1.0,Sri Lanka
9,2003-01-14,India,New Zealand,0.25,0.75,-0.5,New Zealand


In [9]:

# Store previous team results at each venue
venue_history = {}

# Create venue feature columns
features_df["team_1_venue_win_pct"] = np.nan
features_df["team_2_venue_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate venue performance before today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        venue = features_df.at[idx, "venue"]

        team1_history = venue_history.get((team1, venue), [])
        team2_history = venue_history.get((team2, venue), [])

        features_df.at[idx, "team_1_venue_win_pct"] = (
            np.mean(team1_history) if team1_history else 0.5
        )

        features_df.at[idx, "team_2_venue_win_pct"] = (
            np.mean(team2_history) if team2_history else 0.5
        )

    # Update venue history after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        venue = features_df.at[idx, "venue"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):

            team1_result = 1 if winner == team1 else 0
            team2_result = 1 if winner == team2 else 0

            venue_history.setdefault((team1, venue), []).append(team1_result)
            venue_history.setdefault((team2, venue), []).append(team2_result)

# Calculate venue performance difference
features_df["venue_win_pct_difference"] = (
    features_df["team_1_venue_win_pct"] -
    features_df["team_2_venue_win_pct"]
)

print("Venue performance features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2", "venue",
         "team_1_venue_win_pct", "team_2_venue_win_pct",
         "venue_win_pct_difference", "winner"]
    ].head(20)
)


Venue performance features created!


,match_date,team_1,team_2,venue,team_1_venue_win_pct,team_2_venue_win_pct,venue_win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",0.5,0.5,0.0,England
1,2002-06-29,England,India,"Lord's, London",0.5,0.5,0.0,India
2,2002-12-29,New Zealand,India,"McLean Park, Napier",0.5,0.5,0.0,New Zealand
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",0.5,0.5,0.0,New Zealand
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",0.5,0.5,0.0,New Zealand
5,2003-01-08,New Zealand,India,"Westpac Stadium, Wellington",0.5,0.5,0.0,India
6,2003-01-09,Sri Lanka,Australia,Sydney Cricket Ground,0.5,0.5,0.0,Sri Lanka
7,2003-01-11,Australia,England,"Bellerive Oval, Hobart",0.5,0.5,0.0,Australia
8,2003-01-13,Sri Lanka,England,Sydney Cricket Ground,1.0,0.5,0.5,Sri Lanka
9,2003-01-14,India,New Zealand,"Westpac Park, Hamilton",0.5,0.5,0.0,New Zealand


In [10]:

# Inspect unique venues and cities
print("Number of unique venues:", features_df["venue"].nunique())
print("Number of unique cities:", features_df["city"].nunique())

print("\nSample venue-city combinations:")

venue_city_df = (
    features_df[["venue", "city"]]
    .drop_duplicates()
    .sort_values(["city", "venue"])
    .reset_index(drop=True)
)

display(venue_city_df.head(30))

print("\nSample team names:")
print(sorted(set(features_df["team_1"]) | set(features_df["team_2"])))


Number of unique venues: 331
Number of unique cities: 159

Sample venue-city combinations:


,venue,city
0,Mannofield Park,Aberdeen
1,"Mannofield Park, Aberdeen",Aberdeen
2,Sheikh Zayed Stadium,Abu Dhabi
3,"Sheikh Zayed Stadium, Abu Dhabi",Abu Dhabi
4,"Zayed Cricket Stadium, Abu Dhabi",Abu Dhabi
5,Adelaide Oval,Adelaide
6,Narendra Modi Stadium,Ahmedabad
7,"Narendra Modi Stadium, Ahmedabad",Ahmedabad
8,"Sardar Patel (Gujarat) Stadium, Motera",Ahmedabad
9,"Sardar Patel Stadium, Motera",Ahmedabad



Sample team names:
['Afghanistan', 'Australia', 'Bangladesh', 'Bermuda', 'Canada', 'England', 'Hong Kong', 'India', 'Ireland', 'Jersey', 'Kenya', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']


In [11]:

# Display all unique teams
teams = sorted(
    set(features_df["team_1"].unique()) |
    set(features_df["team_2"].unique())
)

print("Unique teams:", len(teams))
print("\nTeams:")
for team in teams:
    print("-", team)

# Display all unique cities
cities = sorted(features_df["city"].dropna().unique())

print("\nUnique cities:", len(cities))
print("\nCities:")
for city in cities:
    print("-", city)


Unique teams: 25

Teams:
- Afghanistan
- Australia
- Bangladesh
- Bermuda
- Canada
- England
- Hong Kong
- India
- Ireland
- Jersey
- Kenya
- Namibia
- Nepal
- Netherlands
- New Zealand
- Oman
- Pakistan
- Papua New Guinea
- Scotland
- South Africa
- Sri Lanka
- United Arab Emirates
- United States of America
- West Indies
- Zimbabwe

Unique cities: 159

Cities:
- Aberdeen
- Abu Dhabi
- Adelaide
- Ahmedabad
- Al Amarat
- Amstelveen
- Antigua
- Auckland
- Ayr
- Bangalore
- Bangi
- Barbados
- Basseterre
- Belfast
- Bengaluru
- Benoni
- Birmingham
- Bloemfontein
- Bogra
- Bready
- Bridgetown
- Brisbane
- Bristol
- Bulawayo
- Cairns
- Canberra
- Canterbury
- Cape Town
- Cardiff
- Centurion
- Chandigarh
- Chattogram
- Chelmsford
- Chennai
- Chester-le-Street
- Chittagong
- Christchurch
- Colombo
- Cuttack
- Dallas
- Dambulla
- Darwin
- Dehradun
- Delhi
- Deventer
- Dhaka
- Dharamsala
- Dharamshala
- Dharmasala
- Dominica
- Dubai
- Dublin
- Dundee
- Dunedin
- Durban
- East London
- Edinburgh

In [12]:

# Map each team to its home country
team_country_map = {
    "Afghanistan": "Afghanistan",
    "Australia": "Australia",
    "Bangladesh": "Bangladesh",
    "Bermuda": "Bermuda",
    "Canada": "Canada",
    "England": "England",
    "Hong Kong": "Hong Kong",
    "India": "India",
    "Ireland": "Ireland",
    "Jersey": "Jersey",
    "Kenya": "Kenya",
    "Namibia": "Namibia",
    "Nepal": "Nepal",
    "Netherlands": "Netherlands",
    "New Zealand": "New Zealand",
    "Oman": "Oman",
    "Pakistan": "Pakistan",
    "Papua New Guinea": "Papua New Guinea",
    "Scotland": "Scotland",
    "South Africa": "South Africa",
    "Sri Lanka": "Sri Lanka",
    "United Arab Emirates": "United Arab Emirates",
    "United States of America": "United States of America",
    "Zimbabwe": "Zimbabwe"
}

# West Indies is not assigned a single country.
# Its home-region treatment will need a separate rule.

# Check whether every team except West Indies has a mapping
unmapped_teams = [
    team for team in teams
    if team not in team_country_map and team != "West Indies"
]

print("Total teams:", len(teams))
print("Mapped teams:", len(team_country_map))
print("Unmapped teams:", unmapped_teams)
print("West Indies requires special handling:", "West Indies" in teams)


Total teams: 25
Mapped teams: 24
Unmapped teams: []
West Indies requires special handling: True


In [13]:

# Divide unique cities into groups of 40 for easier mapping
city_groups = [
    cities[i:i + 40]
    for i in range(0, len(cities), 40)
]

print("Total city groups:", len(city_groups))

for i, group in enumerate(city_groups, start=1):
    print(f"\n--- City Group {i} ---")
    print(", ".join(group))


Total city groups: 4

--- City Group 1 ---
Aberdeen, Abu Dhabi, Adelaide, Ahmedabad, Al Amarat, Amstelveen, Antigua, Auckland, Ayr, Bangalore, Bangi, Barbados, Basseterre, Belfast, Bengaluru, Benoni, Birmingham, Bloemfontein, Bogra, Bready, Bridgetown, Brisbane, Bristol, Bulawayo, Cairns, Canberra, Canterbury, Cape Town, Cardiff, Centurion, Chandigarh, Chattogram, Chelmsford, Chennai, Chester-le-Street, Chittagong, Christchurch, Colombo, Cuttack, Dallas

--- City Group 2 ---
Dambulla, Darwin, Dehradun, Delhi, Deventer, Dhaka, Dharamsala, Dharamshala, Dharmasala, Dominica, Dubai, Dublin, Dundee, Dunedin, Durban, East London, Edinburgh, Faisalabad, Faridabad, Fatullah, Galle, Glasgow, Gqeberha, Greater Noida, Grenada, Gros Islet, Gros Islet, St Lucia, Guwahati, Guyana, Gwalior, Hambantota, Hamilton, Harare, Hobart, Hong Kong, Hyderabad, Indore, Jaipur, Jamaica, Jamshedpur

--- City Group 3 ---
Johannesburg, Kandy, Kanpur, Karachi, Kathmandu, Khulna, Kimberley, King City, Kingston, Kingst

In [14]:

# Standardise known city-name variations
city_name_map = {
    "Bangalore": "Bengaluru",
    "Chittagong": "Chattogram",
    "Dharamsala": "Dharamshala",
    "Dharmasala": "Dharamshala",
    "Sharjah, UAE": "Sharjah",
    "Mirpur, Dhaka": "Mirpur"
}

# Preserve original city and create standardised city column
features_df["city_original"] = features_df["city"]

features_df["city_standard"] = (
    features_df["city"]
    .replace(city_name_map)
)

# Check standardisation
print("Original unique cities:", features_df["city"].nunique())
print("Standardised unique cities:", features_df["city_standard"].nunique())

print("\nChanged city names:")
display(
    features_df.loc[
        features_df["city"] != features_df["city_standard"],
        ["city", "city_standard"]
    ].drop_duplicates().sort_values("city")
)


Original unique cities: 159
Standardised unique cities: 153

Changed city names:


,city,city_standard
493,Bangalore,Bengaluru
730,Chittagong,Chattogram
1091,Dharamsala,Dharamshala
1282,Dharmasala,Dharamshala
1230,"Mirpur, Dhaka",Mirpur
2448,"Sharjah, UAE",Sharjah


In [15]:

# City-to-country mapping: Group 1

city_country_map = {
    "Aberdeen": "Scotland",
    "Abu Dhabi": "United Arab Emirates",
    "Adelaide": "Australia",
    "Ahmedabad": "India",
    "Al Amarat": "Oman",
    "Amstelveen": "Netherlands",
    "Antigua": "Antigua and Barbuda",
    "Auckland": "New Zealand",
    "Ayr": "Scotland",
    "Bengaluru": "India",
    "Bangi": "Malaysia",
    "Barbados": "Barbados",
    "Basseterre": "Saint Kitts and Nevis",
    "Belfast": "Northern Ireland",
    "Benoni": "South Africa",
    "Birmingham": "England",
    "Bloemfontein": "South Africa",
    "Bogra": "Bangladesh",
    "Bready": "Northern Ireland",
    "Bridgetown": "Barbados",
    "Brisbane": "Australia",
    "Bristol": "England",
    "Bulawayo": "Zimbabwe",
    "Cairns": "Australia",
    "Canberra": "Australia",
    "Canterbury": "New Zealand",
    "Cape Town": "South Africa",
    "Cardiff": "Wales",
    "Centurion": "South Africa",
    "Chandigarh": "India",
    "Chattogram": "Bangladesh",
    "Chelmsford": "England",
    "Chennai": "India",
    "Chester-le-Street": "England",
    "Christchurch": "New Zealand",
    "Colombo": "Sri Lanka",
    "Cuttack": "India",
    "Dallas": "United States of America"
}

# Check Group 1 coverage
group_1 = city_groups[0]

missing_group_1 = [
    city for city in group_1
    if city not in city_country_map
]

print("Group 1 cities:", len(group_1))
print("Mapped cities:", len(group_1) - len(missing_group_1))
print("Unmapped cities:", missing_group_1)


Group 1 cities: 40
Mapped cities: 38
Unmapped cities: ['Bangalore', 'Chittagong']


In [16]:

group_1_standard = [
    city_name_map.get(city, city)
    for city in city_groups[0]
]

missing_group_1 = [
    city for city in group_1_standard
    if city not in city_country_map
]

print("Group 1 cities:", len(group_1_standard))
print("Mapped cities:", len(group_1_standard) - len(missing_group_1))
print("Unmapped cities:", missing_group_1)


Group 1 cities: 40
Mapped cities: 40
Unmapped cities: []


In [17]:

# City-to-country mapping: Group 2

city_country_map.update({
    "Dambulla": "Sri Lanka",
    "Darwin": "Australia",
    "Dehradun": "India",
    "Delhi": "India",
    "Deventer": "Netherlands",
    "Dhaka": "Bangladesh",
    "Dharamshala": "India",
    "Dominica": "Dominica",
    "Dubai": "United Arab Emirates",
    "Dublin": "Ireland",
    "Dundee": "Scotland",
    "Dunedin": "New Zealand",
    "Durban": "South Africa",
    "East London": "South Africa",
    "Edinburgh": "Scotland",
    "Faisalabad": "Pakistan",
    "Faridabad": "India",
    "Fatullah": "Bangladesh",
    "Galle": "Sri Lanka",
    "Glasgow": "Scotland",
    "Gqeberha": "South Africa",
    "Greater Noida": "India",
    "Grenada": "Grenada",
    "Gros Islet": "Saint Lucia",
    "Gros Islet, St Lucia": "Saint Lucia",
    "Guwahati": "India",
    "Guyana": "Guyana",
    "Gwalior": "India",
    "Hambantota": "Sri Lanka",
    "Hamilton": "New Zealand",
    "Harare": "Zimbabwe",
    "Hobart": "Australia",
    "Hong Kong": "Hong Kong",
    "Hyderabad": "India",
    "Indore": "India",
    "Jaipur": "India",
    "Jamaica": "Jamaica",
    "Jamshedpur": "India"
})

# Validate Group 2 using standardised city names
group_2_standard = [
    city_name_map.get(city, city)
    for city in city_groups[1]
]

missing_group_2 = [
    city for city in group_2_standard
    if city not in city_country_map
]

print("Group 2 cities:", len(group_2_standard))
print("Mapped cities:", len(group_2_standard) - len(missing_group_2))
print("Unmapped cities:", missing_group_2)


Group 2 cities: 40
Mapped cities: 40
Unmapped cities: []


In [18]:

# City-to-country mapping: Group 3

city_country_map.update({
    "Johannesburg": "South Africa",
    "Kandy": "Sri Lanka",
    "Kanpur": "India",
    "Karachi": "Pakistan",
    "Kathmandu": "Nepal",
    "Khulna": "Bangladesh",
    "Kimberley": "South Africa",
    "King City": "Canada",
    "Kingston": "Jamaica",
    "Kingstown": "Saint Vincent and the Grenadines",
    "Kirtipur": "Nepal",
    "Kochi": "India",
    "Kolkata": "India",
    "Kuala Lumpur": "Malaysia",
    "Lahore": "Pakistan",
    "Lauderhill": "United States of America",
    "Leeds": "England",
    "Lincoln": "New Zealand",
    "London": "England",
    "Lucknow": "India",
    "Mackay": "Australia",
    "Magheramason": "Northern Ireland",
    "Manchester": "England",
    "Margao": "India",
    "Melbourne": "Australia",
    "Mirpur": "Bangladesh",
    "Mombasa": "Kenya",
    "Mount Maunganui": "New Zealand",
    "Multan": "Pakistan",
    "Mumbai": "India",
    "Nagpur": "India",
    "Nairobi": "Kenya",
    "Napier": "New Zealand",
    "Nelson": "New Zealand",
    "New Chandigarh": "India",
    "North Sound": "Antigua and Barbuda",
    "Nottingham": "England",
    "Paarl": "South Africa",
    "Pearland": "United States of America"
})

# Validate Group 3
group_3_standard = [
    city_name_map.get(city, city)
    for city in city_groups[2]
]

missing_group_3 = [
    city for city in group_3_standard
    if city not in city_country_map
]

print("Group 3 cities:", len(group_3_standard))
print("Mapped cities:", len(group_3_standard) - len(missing_group_3))
print("Unmapped cities:", missing_group_3)


Group 3 cities: 40
Mapped cities: 40
Unmapped cities: []


In [19]:

# City-to-country mapping: Group 4

city_country_map.update({
    "Perth": "Australia",
    "Peshawar": "Pakistan",
    "Pietermaritzburg": "South Africa",
    "Port Elizabeth": "South Africa",
    "Port Moresby": "Papua New Guinea",
    "Port of Spain": "Trinidad and Tobago",
    "Potchefstroom": "South Africa",
    "Providence": "Guyana",
    "Pune": "India",
    "Queenstown": "New Zealand",
    "Raipur": "India",
    "Rajkot": "India",
    "Ranchi": "India",
    "Rawalpindi": "Pakistan",
    "Rotterdam": "Netherlands",
    "Sharjah": "United Arab Emirates",
    "Sheikhupura": "Pakistan",
    "Sind": "Pakistan",
    "Southampton": "England",
    "St George's": "Grenada",
    "St Kitts": "Saint Kitts and Nevis",
    "St Lucia": "Saint Lucia",
    "St Vincent": "Saint Vincent and the Grenadines",
    "Sydney": "Australia",
    "Sylhet": "Bangladesh",
    "Tarouba": "Trinidad and Tobago",
    "Taunton": "England",
    "The Hague": "Netherlands",
    "Thiruvananthapuram": "India",
    "Toronto": "Canada",
    "Townsville": "Australia",
    "Trinidad": "Trinidad and Tobago",
    "Utrecht": "Netherlands",
    "Vadodara": "India",
    "Visakhapatnam": "India",
    "Wellington": "New Zealand",
    "Whangarei": "New Zealand",
    "Windhoek": "Namibia"
})

# Validate Group 4
group_4_standard = [
    city_name_map.get(city, city)
    for city in city_groups[3]
]

missing_group_4 = [
    city for city in group_4_standard
    if city not in city_country_map
]

print("Group 4 cities:", len(group_4_standard))
print("Mapped cities:", len(group_4_standard) - len(missing_group_4))
print("Unmapped cities:", missing_group_4)


Group 4 cities: 39
Mapped cities: 39
Unmapped cities: []


In [20]:

# Get all standardised cities from the dataset
all_standard_cities = sorted(
    features_df["city_standard"].dropna().unique()
)

# Find cities missing from our mapping
unmapped_cities = [
    city for city in all_standard_cities
    if city not in city_country_map
]

# Find mapping entries not used in the dataset
unused_mappings = [
    city for city in city_country_map
    if city not in all_standard_cities
]

# Create mapping validation table
city_mapping_df = pd.DataFrame({
    "city": all_standard_cities,
    "country": [
        city_country_map.get(city, "UNMAPPED")
        for city in all_standard_cities
    ]
})

print("Total standardised cities:", len(all_standard_cities))
print("Mapped cities:", len(all_standard_cities) - len(unmapped_cities))
print("Unmapped cities:", unmapped_cities)
print("Unused mapping entries:", unused_mappings)

print("\nComplete city-country mapping:")
display(city_mapping_df.sort_values("country"))


Total standardised cities: 153
Mapped cities: 153
Unmapped cities: []
Unused mapping entries: []

Complete city-country mapping:


,city,country
6,Antigua,Antigua and Barbuda
111,North Sound,Antigua and Barbuda
138,Sydney,Australia
115,Perth,Australia
2,Adelaide,Australia
...,...,...
114,Pearland,United States of America
37,Dallas,United States of America
27,Cardiff,Wales
22,Bulawayo,Zimbabwe


In [21]:

# Map each match's standardised city to its country
features_df["venue_country"] = features_df["city_standard"].map(
    city_country_map
)

# Validate venue-country assignment
print("Missing venue countries:", features_df["venue_country"].isna().sum())

print("\nVenue country distribution:")
print(features_df["venue_country"].value_counts())

print("\nSample match records:")
display(
    features_df[
        ["match_date", "team_1", "team_2",
         "city_standard", "venue_country"]
    ].head(20)
)


Missing venue countries: 0

Venue country distribution:
venue_country
India                               288
England                             266
South Africa                        232
Australia                           225
Sri Lanka                           218
Zimbabwe                            213
United Arab Emirates                199
New Zealand                         184
Bangladesh                          181
Pakistan                            114
Namibia                              48
Nepal                                47
Scotland                             45
Ireland                              42
Netherlands                          40
Oman                                 39
United States of America             34
Northern Ireland                     34
Barbados                             33
Trinidad and Tobago                  32
Guyana                               28
Wales                                28
Antigua and Barbuda                  26
Jamaica   

,match_date,team_1,team_2,city_standard,venue_country
0,2002-06-27,England,Sri Lanka,Nottingham,England
1,2002-06-29,England,India,London,England
2,2002-12-29,New Zealand,India,Napier,New Zealand
3,2003-01-01,India,New Zealand,Christchurch,New Zealand
4,2003-01-04,India,New Zealand,Queenstown,New Zealand
5,2003-01-08,New Zealand,India,Wellington,New Zealand
6,2003-01-09,Sri Lanka,Australia,Sydney,Australia
7,2003-01-11,Australia,England,Hobart,Australia
8,2003-01-13,Sri Lanka,England,Sydney,Australia
9,2003-01-14,India,New Zealand,Hamilton,New Zealand


In [22]:

# Map each match's standardised city to its country
features_df["venue_country"] = features_df["city_standard"].map(
    city_country_map
)

# Validate venue-country assignment
print("Missing venue countries:", features_df["venue_country"].isna().sum())

print("\nVenue country distribution:")
print(features_df["venue_country"].value_counts())

print("\nSample match records:")
display(
    features_df[
        ["match_date", "team_1", "team_2",
         "city_standard", "venue_country"]
    ].head(20)
)


Missing venue countries: 0

Venue country distribution:
venue_country
India                               288
England                             266
South Africa                        232
Australia                           225
Sri Lanka                           218
Zimbabwe                            213
United Arab Emirates                199
New Zealand                         184
Bangladesh                          181
Pakistan                            114
Namibia                              48
Nepal                                47
Scotland                             45
Ireland                              42
Netherlands                          40
Oman                                 39
United States of America             34
Northern Ireland                     34
Barbados                             33
Trinidad and Tobago                  32
Guyana                               28
Wales                                28
Antigua and Barbuda                  26
Jamaica   

,match_date,team_1,team_2,city_standard,venue_country
0,2002-06-27,England,Sri Lanka,Nottingham,England
1,2002-06-29,England,India,London,England
2,2002-12-29,New Zealand,India,Napier,New Zealand
3,2003-01-01,India,New Zealand,Christchurch,New Zealand
4,2003-01-04,India,New Zealand,Queenstown,New Zealand
5,2003-01-08,New Zealand,India,Wellington,New Zealand
6,2003-01-09,Sri Lanka,Australia,Sydney,Australia
7,2003-01-11,Australia,England,Hobart,Australia
8,2003-01-13,Sri Lanka,England,Sydney,Australia
9,2003-01-14,India,New Zealand,Hamilton,New Zealand


In [23]:

# Define home territories for cricket teams
team_home_territories = {
    **team_country_map,
    
    # West Indies represents a regional cricket team.
    "West Indies": [
        "Antigua and Barbuda",
        "Barbados",
        "Dominica",
        "Grenada",
        "Guyana",
        "Jamaica",
        "Saint Kitts and Nevis",
        "Saint Lucia",
        "Saint Vincent and the Grenadines",
        "Trinidad and Tobago"
    ]
}

print("Total teams with home-territory rules:",
      len(team_home_territories))

print("\nWest Indies home territories:")
print(team_home_territories["West Indies"])

# Check that every team has a home-territory rule
missing_home_rules = [
    team for team in teams
    if team not in team_home_territories
]

print("\nTeams without home-territory rules:", missing_home_rules)


Total teams with home-territory rules: 25

West Indies home territories:
['Antigua and Barbuda', 'Barbados', 'Dominica', 'Grenada', 'Guyana', 'Jamaica', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Trinidad and Tobago']

Teams without home-territory rules: []


In [24]:

# Check whether a team considers the venue its home territory
def is_home_match(team, venue_country):
    home_territory = team_home_territories[team]

    if isinstance(home_territory, list):
        return venue_country in home_territory

    return venue_country == home_territory


# Identify whether Team 1 is playing at home
features_df["team_1_is_home"] = features_df.apply(
    lambda row: int(is_home_match(row["team_1"], row["venue_country"])),
    axis=1
)

# Identify whether Team 2 is playing at home
features_df["team_2_is_home"] = features_df.apply(
    lambda row: int(is_home_match(row["team_2"], row["venue_country"])),
    axis=1
)

# Identify neutral venues
features_df["neutral_venue"] = (
    (features_df["team_1_is_home"] == 0) &
    (features_df["team_2_is_home"] == 0)
).astype(int)

# Identify which team has home advantage
features_df["home_advantage"] = np.select(
    [
        features_df["team_1_is_home"] == 1,
        features_df["team_2_is_home"] == 1
    ],
    [
        1,
        -1
    ],
    default=0
)

print("Home and neutral venue features created!")

print("\nHome advantage distribution:")
print(features_df["home_advantage"].value_counts())

print("\nNeutral venue matches:",
      features_df["neutral_venue"].sum())

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "venue_country", "team_1_is_home",
         "team_2_is_home", "neutral_venue",
         "home_advantage"]
    ].head(25)
)


Home and neutral venue features created!

Home advantage distribution:
home_advantage
 1    1538
 0     748
-1     467
Name: count, dtype: int64

Neutral venue matches: 748


,match_date,team_1,team_2,venue_country,team_1_is_home,team_2_is_home,neutral_venue,home_advantage
0,2002-06-27,England,Sri Lanka,England,1,0,0,1
1,2002-06-29,England,India,England,1,0,0,1
2,2002-12-29,New Zealand,India,New Zealand,1,0,0,1
3,2003-01-01,India,New Zealand,New Zealand,0,1,0,-1
4,2003-01-04,India,New Zealand,New Zealand,0,1,0,-1
5,2003-01-08,New Zealand,India,New Zealand,1,0,0,1
6,2003-01-09,Sri Lanka,Australia,Australia,0,1,0,-1
7,2003-01-11,Australia,England,Australia,1,0,0,1
8,2003-01-13,Sri Lanka,England,Australia,0,0,1,0
9,2003-01-14,India,New Zealand,New Zealand,0,1,0,-1


In [25]:

# Check matches where both teams are classified as home
both_home = features_df[
    (features_df["team_1_is_home"] == 1) &
    (features_df["team_2_is_home"] == 1)
]

# Check matches where neither team is classified as home
both_away = features_df[
    (features_df["team_1_is_home"] == 0) &
    (features_df["team_2_is_home"] == 0)
]

# Check consistency of neutral venue indicator
neutral_check = (
    features_df["neutral_venue"] ==
    ((features_df["team_1_is_home"] == 0) &
     (features_df["team_2_is_home"] == 0)).astype(int)
)

print("Both teams classified as home:", len(both_home))
print("Neither team classified as home:", len(both_away))
print("Neutral indicator mismatches:", (~neutral_check).sum())

print("\nMatches where both teams are classified as home:")
display(
    both_home[
        ["match_date", "team_1", "team_2", "venue_country",
         "team_1_is_home", "team_2_is_home", "home_advantage"]
    ].head(20)
)

print("\nHome advantage distribution:")
print(features_df["home_advantage"].value_counts())


Both teams classified as home: 0
Neither team classified as home: 748
Neutral indicator mismatches: 0

Matches where both teams are classified as home:


,match_date,team_1,team_2,venue_country,team_1_is_home,team_2_is_home,home_advantage



Home advantage distribution:
home_advantage
 1    1538
 0     748
-1     467
Name: count, dtype: int64


In [26]:

# Store historical home and away results for each team
home_results = {}
away_results = {}

# Create feature columns
features_df["team_1_home_win_pct"] = np.nan
features_df["team_2_home_win_pct"] = np.nan
features_df["team_1_away_win_pct"] = np.nan
features_df["team_2_away_win_pct"] = np.nan

# Process matches chronologically, date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate historical percentages before today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        # Team 1 home and away history
        t1_home = home_results.get(team1, [])
        t1_away = away_results.get(team1, [])

        # Team 2 home and away history
        t2_home = home_results.get(team2, [])
        t2_away = away_results.get(team2, [])

        features_df.at[idx, "team_1_home_win_pct"] = (
            np.mean(t1_home) if t1_home else 0.5
        )

        features_df.at[idx, "team_1_away_win_pct"] = (
            np.mean(t1_away) if t1_away else 0.5
        )

        features_df.at[idx, "team_2_home_win_pct"] = (
            np.mean(t2_home) if t2_home else 0.5
        )

        features_df.at[idx, "team_2_away_win_pct"] = (
            np.mean(t2_away) if t2_away else 0.5
        )

    # Update history only after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.isna(winner):
            continue

        team1_result = int(winner == team1)
        team2_result = int(winner == team2)

        team1_home = features_df.at[idx, "team_1_is_home"]
        team2_home = features_df.at[idx, "team_2_is_home"]

        # Record Team 1's result
        if team1_home:
            home_results.setdefault(team1, []).append(team1_result)
        elif not features_df.at[idx, "neutral_venue"]:
            away_results.setdefault(team1, []).append(team1_result)

        # Record Team 2's result
        if team2_home:
            home_results.setdefault(team2, []).append(team2_result)
        elif not features_df.at[idx, "neutral_venue"]:
            away_results.setdefault(team2, []).append(team2_result)

print("Historical home and away performance features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_home_win_pct", "team_1_away_win_pct",
         "team_2_home_win_pct", "team_2_away_win_pct",
         "home_advantage", "winner"]
    ].head(20)
)


Historical home and away performance features created!


,match_date,team_1,team_2,team_1_home_win_pct,team_1_away_win_pct,team_2_home_win_pct,team_2_away_win_pct,home_advantage,winner
0,2002-06-27,England,Sri Lanka,0.500000,0.500000,0.500000,0.500000,1,England
1,2002-06-29,England,India,1.000000,0.500000,0.500000,0.500000,1,India
2,2002-12-29,New Zealand,India,0.500000,0.500000,0.500000,1.000000,1,New Zealand
3,2003-01-01,India,New Zealand,0.500000,0.500000,1.000000,0.500000,-1,New Zealand
4,2003-01-04,India,New Zealand,0.500000,0.333333,1.000000,0.500000,-1,New Zealand
5,2003-01-08,New Zealand,India,1.000000,0.500000,0.500000,0.250000,1,India
6,2003-01-09,Sri Lanka,Australia,0.500000,0.000000,0.500000,0.500000,-1,Sri Lanka
7,2003-01-11,Australia,England,0.000000,0.500000,0.500000,0.500000,1,Australia
8,2003-01-13,Sri Lanka,England,0.500000,0.500000,0.500000,0.000000,0,Sri Lanka
9,2003-01-14,India,New Zealand,0.500000,0.400000,0.750000,0.500000,-1,New Zealand


In [27]:

# Compare historical home and away performance between the teams

features_df["home_win_pct_difference"] = (
    features_df["team_1_home_win_pct"] -
    features_df["team_2_home_win_pct"]
)

features_df["away_win_pct_difference"] = (
    features_df["team_1_away_win_pct"] -
    features_df["team_2_away_win_pct"]
)

print("Home and away performance difference features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "home_win_pct_difference",
         "away_win_pct_difference",
         "home_advantage"]
    ].head(20)
)


Home and away performance difference features created!


,match_date,team_1,team_2,home_win_pct_difference,away_win_pct_difference,home_advantage
0,2002-06-27,England,Sri Lanka,0.000000,0.000000,1
1,2002-06-29,England,India,0.500000,0.000000,1
2,2002-12-29,New Zealand,India,0.000000,-0.500000,1
3,2003-01-01,India,New Zealand,-0.500000,0.000000,-1
4,2003-01-04,India,New Zealand,-0.500000,-0.166667,-1
5,2003-01-08,New Zealand,India,0.500000,0.250000,1
6,2003-01-09,Sri Lanka,Australia,0.000000,-0.500000,-1
7,2003-01-11,Australia,England,-0.500000,0.000000,1
8,2003-01-13,Sri Lanka,England,0.000000,0.500000,0
9,2003-01-14,India,New Zealand,-0.250000,-0.100000,-1


In [28]:

home_away_cols = [
    "team_1_home_win_pct",
    "team_2_home_win_pct",
    "team_1_away_win_pct",
    "team_2_away_win_pct",
    "home_win_pct_difference",
    "away_win_pct_difference"
]

print("Missing values:")
print(features_df[home_away_cols].isnull().sum())

print("\nPercentage range validation:")
percentage_cols = [
    "team_1_home_win_pct",
    "team_2_home_win_pct",
    "team_1_away_win_pct",
    "team_2_away_win_pct"
]

for col in percentage_cols:
    print(
        f"{col}:",
        features_df[col].between(0, 1).all()
    )

print("\nDifference validation:")
print(
    "Home difference correct:",
    np.allclose(
        features_df["home_win_pct_difference"],
        features_df["team_1_home_win_pct"] -
        features_df["team_2_home_win_pct"]
    )
)

print(
    "Away difference correct:",
    np.allclose(
        features_df["away_win_pct_difference"],
        features_df["team_1_away_win_pct"] -
        features_df["team_2_away_win_pct"]
    )
)


Missing values:
team_1_home_win_pct        0
team_2_home_win_pct        0
team_1_away_win_pct        0
team_2_away_win_pct        0
home_win_pct_difference    0
away_win_pct_difference    0
dtype: int64

Percentage range validation:
team_1_home_win_pct: True
team_2_home_win_pct: True
team_1_away_win_pct: True
team_2_away_win_pct: True

Difference validation:
Home difference correct: True
Away difference correct: True


In [29]:

feature_columns = [
    "team_1_win_pct",
    "team_2_win_pct",
    "win_pct_difference",
    "team_1_recent_form",
    "team_2_recent_form",
    "recent_form_difference",
    "team_1_h2h_win_pct",
    "team_2_h2h_win_pct",
    "h2h_win_pct_difference",
    "team_1_venue_win_pct",
    "team_2_venue_win_pct",
    "venue_win_pct_difference",
    "team_1_is_home",
    "team_2_is_home",
    "neutral_venue",
    "home_advantage",
    "team_1_home_win_pct",
    "team_2_home_win_pct",
    "team_1_away_win_pct",
    "team_2_away_win_pct",
    "home_win_pct_difference",
    "away_win_pct_difference"
]

print("Total engineered features:", len(feature_columns))
print("\nFeature availability:")
print(features_df[feature_columns].notna().sum())

print("\nFeature dataset shape:", features_df.shape)


Total engineered features: 22

Feature availability:
team_1_win_pct              2753
team_2_win_pct              2753
win_pct_difference          2753
team_1_recent_form          2753
team_2_recent_form          2753
recent_form_difference      2753
team_1_h2h_win_pct          2753
team_2_h2h_win_pct          2753
h2h_win_pct_difference      2753
team_1_venue_win_pct        2753
team_2_venue_win_pct        2753
venue_win_pct_difference    2753
team_1_is_home              2753
team_2_is_home              2753
neutral_venue               2753
home_advantage              2753
team_1_home_win_pct         2753
team_2_home_win_pct         2753
team_1_away_win_pct         2753
team_2_away_win_pct         2753
home_win_pct_difference     2753
away_win_pct_difference     2753
dtype: int64

Feature dataset shape: (2753, 37)


In [30]:

# Inspect matches involving Ireland or England
geo_audit = features_df[
    features_df["team_1"].isin(["Ireland", "England"]) |
    features_df["team_2"].isin(["Ireland", "England"])
][
    ["match_date", "team_1", "team_2",
     "city_standard", "venue_country",
     "team_1_is_home", "team_2_is_home",
     "neutral_venue", "home_advantage"]
].copy()

display(
    geo_audit[
        geo_audit["venue_country"].isin(
            ["Ireland", "Northern Ireland", "Wales", "England"]
        )
    ].tail(30)
)


,match_date,team_1,team_2,city_standard,venue_country,team_1_is_home,team_2_is_home,neutral_venue,home_advantage
2347,2023-09-10,England,New Zealand,Southampton,England,1,0,0,1
2350,2023-09-13,England,New Zealand,London,England,1,0,0,1
2354,2023-09-15,England,New Zealand,London,England,1,0,0,1
2360,2023-09-23,England,Ireland,Nottingham,England,1,0,0,1
2362,2023-09-26,England,Ireland,Bristol,England,1,0,0,1
2473,2024-09-19,England,Australia,Nottingham,England,1,0,0,1
2477,2024-09-21,Australia,England,Leeds,England,0,1,0,-1
2482,2024-09-24,Australia,England,Chester-le-Street,England,0,1,0,-1
2485,2024-09-27,England,Australia,London,England,1,0,0,1
2486,2024-09-29,England,Australia,Bristol,England,1,0,0,1


In [31]:

# Correct home territory definitions

team_home_territories["Ireland"] = [
    "Ireland",
    "Northern Ireland"
]

team_home_territories["England"] = [
    "England",
    "Wales"
]

print("Updated home territory rules:")
print("Ireland:", team_home_territories["Ireland"])
print("England:", team_home_territories["England"])


Updated home territory rules:
Ireland: ['Ireland', 'Northern Ireland']
England: ['England', 'Wales']


In [32]:

# Recalculate home-team indicators

features_df["team_1_is_home"] = [
    int(is_home_match(team, country))
    for team, country in zip(
        features_df["team_1"],
        features_df["venue_country"]
    )
]

features_df["team_2_is_home"] = [
    int(is_home_match(team, country))
    for team, country in zip(
        features_df["team_2"],
        features_df["venue_country"]
    )
]

features_df["neutral_venue"] = (
    (features_df["team_1_is_home"] == 0) &
    (features_df["team_2_is_home"] == 0)
).astype(int)

features_df["home_advantage"] = np.select(
    [
        features_df["team_1_is_home"] == 1,
        features_df["team_2_is_home"] == 1
    ],
    [1, -1],
    default=0
)

print("Home advantage features recalculated!")

display(
    features_df[
        features_df["venue_country"].isin(
            ["Ireland", "Northern Ireland", "England", "Wales"]
        )
    ][
        ["match_date", "team_1", "team_2",
         "city_standard", "venue_country",
         "team_1_is_home", "team_2_is_home",
         "neutral_venue", "home_advantage"]
    ].tail(30)
)


Home advantage features recalculated!


,match_date,team_1,team_2,city_standard,venue_country,team_1_is_home,team_2_is_home,neutral_venue,home_advantage
2347,2023-09-10,England,New Zealand,Southampton,England,1,0,0,1
2350,2023-09-13,England,New Zealand,London,England,1,0,0,1
2354,2023-09-15,England,New Zealand,London,England,1,0,0,1
2360,2023-09-23,England,Ireland,Nottingham,England,1,0,0,1
2362,2023-09-26,England,Ireland,Bristol,England,1,0,0,1
2473,2024-09-19,England,Australia,Nottingham,England,1,0,0,1
2477,2024-09-21,Australia,England,Leeds,England,0,1,0,-1
2482,2024-09-24,Australia,England,Chester-le-Street,England,0,1,0,-1
2485,2024-09-27,England,Australia,London,England,1,0,0,1
2486,2024-09-29,England,Australia,Bristol,England,1,0,0,1


In [33]:

# Reset historical home and away results
home_results = {}
away_results = {}

# Reset historical percentage columns
history_cols = [
    "team_1_home_win_pct",
    "team_2_home_win_pct",
    "team_1_away_win_pct",
    "team_2_away_win_pct"
]

features_df[history_cols] = np.nan

# Process matches chronologically
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # First calculate features using only previous dates
    for idx in daily_matches.index:
        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        t1_home = home_results.get(team1, [])
        t1_away = away_results.get(team1, [])
        t2_home = home_results.get(team2, [])
        t2_away = away_results.get(team2, [])

        features_df.at[idx, "team_1_home_win_pct"] = (
            np.mean(t1_home) if t1_home else 0.5
        )
        features_df.at[idx, "team_1_away_win_pct"] = (
            np.mean(t1_away) if t1_away else 0.5
        )
        features_df.at[idx, "team_2_home_win_pct"] = (
            np.mean(t2_home) if t2_home else 0.5
        )
        features_df.at[idx, "team_2_away_win_pct"] = (
            np.mean(t2_away) if t2_away else 0.5
        )

    # Update historical records after all matches on this date
    for idx in daily_matches.index:
        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.isna(winner):
            continue

        team1_result = int(winner == team1)
        team2_result = int(winner == team2)

        team1_home = features_df.at[idx, "team_1_is_home"]
        team2_home = features_df.at[idx, "team_2_is_home"]
        neutral = features_df.at[idx, "neutral_venue"]

        if team1_home:
            home_results.setdefault(team1, []).append(team1_result)
        elif not neutral:
            away_results.setdefault(team1, []).append(team1_result)

        if team2_home:
            home_results.setdefault(team2, []).append(team2_result)
        elif not neutral:
            away_results.setdefault(team2, []).append(team2_result)

# Recalculate differences
features_df["home_win_pct_difference"] = (
    features_df["team_1_home_win_pct"] -
    features_df["team_2_home_win_pct"]
)

features_df["away_win_pct_difference"] = (
    features_df["team_1_away_win_pct"] -
    features_df["team_2_away_win_pct"]
)

print("Historical home/away statistics recalculated!")
print("Home records:", len(home_results))
print("Away records:", len(away_results))


Historical home/away statistics recalculated!
Home records: 22
Away records: 25


In [34]:

# Validate recalculated home/away features

print("Missing values:")
print(features_df[history_cols].isnull().sum())

print("\nPercentage range checks:")
for col in history_cols:
    print(f"{col}: {features_df[col].between(0, 1).all()}")

print("\nHome advantage consistency:")
print(
    "Neutral venue consistency:",
    (
        features_df["neutral_venue"] ==
        (
            (features_df["team_1_is_home"] == 0) &
            (features_df["team_2_is_home"] == 0)
        ).astype(int)
    ).all()
)

print(
    "Home advantage consistency:",
    (
        features_df["home_advantage"] ==
        features_df["team_1_is_home"] -
        features_df["team_2_is_home"]
    ).all()
)


Missing values:
team_1_home_win_pct    0
team_2_home_win_pct    0
team_1_away_win_pct    0
team_2_away_win_pct    0
dtype: int64

Percentage range checks:
team_1_home_win_pct: True
team_2_home_win_pct: True
team_1_away_win_pct: True
team_2_away_win_pct: True

Home advantage consistency:
Neutral venue consistency: True
Home advantage consistency: True


In [35]:

print("Home/neutral venue distribution:")
print(
    features_df["home_advantage"]
    .value_counts()
    .rename(index={
        1: "Team 1 home",
        -1: "Team 2 home",
        0: "Neutral venue"
    })
)

print("\nHome territory classification:")
print("Team 1 home matches:", features_df["team_1_is_home"].sum())
print("Team 2 home matches:", features_df["team_2_is_home"].sum())
print("Neutral matches:", features_df["neutral_venue"].sum())

print(
    "\nTotal matches classified:",
    (
        features_df["team_1_is_home"].sum()
        + features_df["team_2_is_home"].sum()
        + features_df["neutral_venue"].sum()
    )
)


Home/neutral venue distribution:
home_advantage
Team 1 home      1566
Neutral venue     700
Team 2 home       487
Name: count, dtype: int64

Home territory classification:
Team 1 home matches: 1566
Team 2 home matches: 487
Neutral matches: 700

Total matches classified: 2753


In [36]:

# Elo rating configuration

INITIAL_ELO = 1500
K_FACTOR = 32

# Initialize every team with the same rating
teams = set(features_df["team_1"]).union(
    set(features_df["team_2"])
)

elo_ratings = {
    team: INITIAL_ELO
    for team in teams
}

print("Elo rating system initialized!")
print("Initial rating:", INITIAL_ELO)
print("K-factor:", K_FACTOR)
print("Number of teams:", len(elo_ratings))

print("\nSample initial ratings:")
for team in sorted(elo_ratings)[:10]:
    print(f"{team}: {elo_ratings[team]}")


Elo rating system initialized!
Initial rating: 1500
K-factor: 32
Number of teams: 25

Sample initial ratings:
Afghanistan: 1500
Australia: 1500
Bangladesh: 1500
Bermuda: 1500
Canada: 1500
England: 1500
Hong Kong: 1500
India: 1500
Ireland: 1500
Jersey: 1500


In [37]:

def calculate_expected_score(rating_a, rating_b):
    """
    Calculate the expected score for Team A
    based on the Elo rating difference.
    """
    expected_a = 1 / (
        1 + 10 ** ((rating_b - rating_a) / 400)
    )

    return expected_a


# Test the Elo formula
print("Equal ratings (1500 vs 1500):")
print(calculate_expected_score(1500, 1500))

print("\nStronger team (1700 vs 1500):")
print(calculate_expected_score(1700, 1500))

print("\nWeaker team (1300 vs 1500):")
print(calculate_expected_score(1300, 1500))


Equal ratings (1500 vs 1500):
0.5

Stronger team (1700 vs 1500):
0.7597469266479578

Weaker team (1300 vs 1500):
0.2402530733520421


In [38]:

# Initialize Elo feature columns
features_df["team_1_elo"] = np.nan
features_df["team_2_elo"] = np.nan
features_df["elo_difference"] = np.nan

# Reset all teams to their initial ratings
elo_ratings = {
    team: INITIAL_ELO
    for team in teams
}

# Process matches chronologically
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Step 1: Record pre-match Elo ratings for every match on this date
    for idx in daily_matches.index:
        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        rating1 = elo_ratings[team1]
        rating2 = elo_ratings[team2]

        features_df.at[idx, "team_1_elo"] = rating1
        features_df.at[idx, "team_2_elo"] = rating2
        features_df.at[idx, "elo_difference"] = rating1 - rating2

    # Step 2: Update ratings only after all same-day
    # pre-match ratings have been recorded
    for idx in daily_matches.index:
        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        result_type = features_df.at[idx, "result_type"]
        winner = features_df.at[idx, "winner"]

        if result_type == "No result":
            continue

        rating1 = elo_ratings[team1]
        rating2 = elo_ratings[team2]

        expected1 = calculate_expected_score(rating1, rating2)
        expected2 = 1 - expected1

        if result_type == "Tie":
            actual1 = 0.5
            actual2 = 0.5
        else:
            actual1 = float(winner == team1)
            actual2 = float(winner == team2)

        new_rating1 = rating1 + K_FACTOR * (actual1 - expected1)
        new_rating2 = rating2 + K_FACTOR * (actual2 - expected2)

        elo_ratings[team1] = new_rating1
        elo_ratings[team2] = new_rating2

print("Historical Elo features created!")

print("\nFinal Elo ratings:")
for team, rating in sorted(
    elo_ratings.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{team}: {rating:.2f}")

print("\nSample match features:")
display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_elo", "team_2_elo",
         "elo_difference", "result_type", "winner"]
    ].head(20)
)


Historical Elo features created!

Final Elo ratings:
India: 1773.98
New Zealand: 1746.37
Pakistan: 1721.88
Australia: 1689.29
South Africa: 1686.46
Afghanistan: 1649.92
Sri Lanka: 1632.68
England: 1618.44
West Indies: 1552.37
Bangladesh: 1547.35
Scotland: 1519.32
United States of America: 1490.36
Netherlands: 1464.45
Jersey: 1450.12
Nepal: 1424.75
Zimbabwe: 1420.02
Namibia: 1417.64
Hong Kong: 1416.11
Ireland: 1388.84
Bermuda: 1381.55
Kenya: 1365.46
Oman: 1325.32
United Arab Emirates: 1290.42
Canada: 1288.47
Papua New Guinea: 1238.43

Sample match features:


,match_date,team_1,team_2,team_1_elo,team_2_elo,elo_difference,result_type,winner
0,2002-06-27,England,Sri Lanka,1500.000000,1500.000000,0.000000,Win,England
1,2002-06-29,England,India,1516.000000,1500.000000,16.000000,Win,India
2,2002-12-29,New Zealand,India,1500.000000,1516.736307,-16.736307,Win,New Zealand
3,2003-01-01,India,New Zealand,1499.966167,1516.770140,-16.803973,Win,New Zealand
4,2003-01-04,India,New Zealand,1484.739416,1531.996891,-47.257475,Win,New Zealand
5,2003-01-08,New Zealand,India,1545.833926,1470.902380,74.931546,Win,India
6,2003-01-09,Sri Lanka,Australia,1484.000000,1500.000000,-16.000000,Win,Sri Lanka
7,2003-01-11,Australia,England,1483.263693,1499.263693,-16.000000,Win,Australia
8,2003-01-13,Sri Lanka,England,1500.736307,1482.527386,18.208920,Win,Sri Lanka
9,2003-01-14,India,New Zealand,1490.300581,1526.435726,-36.135146,Win,New Zealand


In [39]:

# Reset Elo ratings and feature columns

elo_ratings = {
    team: INITIAL_ELO
    for team in teams
}

features_df[[
    "team_1_elo",
    "team_2_elo",
    "elo_difference"
]] = np.nan

# Process matches by calendar date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Snapshot ratings at the beginning of the date
    starting_ratings = elo_ratings.copy()

    # Store all rating changes for this date
    daily_rating_changes = {
        team: 0.0 for team in teams
    }

    # Calculate features and rating changes
    for idx in daily_matches.index:
        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        rating1 = starting_ratings[team1]
        rating2 = starting_ratings[team2]

        features_df.at[idx, "team_1_elo"] = rating1
        features_df.at[idx, "team_2_elo"] = rating2
        features_df.at[idx, "elo_difference"] = rating1 - rating2

        result_type = features_df.at[idx, "result_type"]
        winner = features_df.at[idx, "winner"]

        if result_type == "No result":
            continue

        expected1 = calculate_expected_score(rating1, rating2)
        expected2 = 1 - expected1

        if result_type == "Tie":
            actual1 = 0.5
            actual2 = 0.5
        else:
            actual1 = float(winner == team1)
            actual2 = float(winner == team2)

        change1 = K_FACTOR * (actual1 - expected1)
        change2 = K_FACTOR * (actual2 - expected2)

        daily_rating_changes[team1] += change1
        daily_rating_changes[team2] += change2

    # Apply all daily changes simultaneously
    for team in teams:
        elo_ratings[team] += daily_rating_changes[team]

print("Corrected Elo calculations completed!")

print("\nFinal Elo ratings:")
for team, rating in sorted(
    elo_ratings.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{team}: {rating:.2f}")

print("\nSame-day sample:")
display(
    features_df[
        features_df["match_date"] == "2003-02-10"
    ][
        ["match_date", "team_1", "team_2",
         "team_1_elo", "team_2_elo", "elo_difference"]
    ]
)


Corrected Elo calculations completed!

Final Elo ratings:
India: 1773.98
New Zealand: 1746.37
Pakistan: 1721.88
Australia: 1689.29
South Africa: 1686.46
Afghanistan: 1649.92
Sri Lanka: 1632.68
England: 1618.44
West Indies: 1552.37
Bangladesh: 1547.35
Scotland: 1519.32
United States of America: 1490.36
Netherlands: 1464.45
Jersey: 1450.12
Nepal: 1424.75
Zimbabwe: 1420.02
Namibia: 1417.64
Hong Kong: 1416.11
Ireland: 1388.84
Bermuda: 1381.55
Kenya: 1365.46
Oman: 1325.32
United Arab Emirates: 1290.42
Canada: 1288.47
Papua New Guinea: 1238.43

Same-day sample:


,match_date,team_1,team_2,team_1_elo,team_2_elo,elo_difference
17,2003-02-10,Sri Lanka,New Zealand,1467.9734,1540.777616,-72.804216
18,2003-02-10,Zimbabwe,Namibia,1500.0000,1500.000000,0.000000


In [40]:

elo_cols = [
    "team_1_elo",
    "team_2_elo",
    "elo_difference"
]

print("Missing values:")
print(features_df[elo_cols].isnull().sum())

print("\nElo difference validation:")
print(
    np.allclose(
        features_df["elo_difference"],
        features_df["team_1_elo"] -
        features_df["team_2_elo"]
    )
)

print("\nRating range:")
print("Minimum Team 1 Elo:", features_df["team_1_elo"].min())
print("Maximum Team 1 Elo:", features_df["team_1_elo"].max())
print("Minimum Team 2 Elo:", features_df["team_2_elo"].min())
print("Maximum Team 2 Elo:", features_df["team_2_elo"].max())

print("\nAll Elo values finite:")
print(
    np.isfinite(features_df[elo_cols].to_numpy()).all()
)


Missing values:
team_1_elo        0
team_2_elo        0
elo_difference    0
dtype: int64

Elo difference validation:
True

Rating range:
Minimum Team 1 Elo: 1180.4623231287142
Maximum Team 1 Elo: 1870.5230713063168
Minimum Team 2 Elo: 1182.1732982993217
Maximum Team 2 Elo: 1865.1822956862306

All Elo values finite:
True


In [41]:

import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_DIR = Path(r"C:\CricIQ")
DATA_PATH = PROJECT_DIR / "data" / "processed" / "integrated" / "cricketiq_integrated_matches.csv"
FEATURE_DIR = PROJECT_DIR / "data" / "features"

# Create feature output directory if it doesn't exist
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Load integrated match dataset
matches_df = pd.read_csv(DATA_PATH)

# Convert match_date into datetime format
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"])

print("Dataset loaded successfully!")
print("Shape:", matches_df.shape)

print("\nColumns:")
print(matches_df.columns.tolist())

print("\nDate range:")
print(matches_df["match_date"].min(), "to", matches_df["match_date"].max())

print("\nResult distribution:")
print(matches_df["result_type"].value_counts(dropna=False))

print("\nSource distribution:")
print(matches_df["source"].value_counts(dropna=False))

print("\nFirst 5 records:")
display(matches_df.head())


Dataset loaded successfully!
Shape: (2753, 11)

Columns:
['match_date', 'team_1', 'team_2', 'venue', 'city', 'toss_winner', 'toss_decision', 'winner', 'result_type', 'source', 'match_id']

Date range:
2002-06-27 00:00:00 to 2026-10-25 00:00:00

Result distribution:
result_type
Win          2614
No result     111
Tie            28
Name: count, dtype: int64

Source distribution:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

First 5 records:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",Nottingham,England,bat,England,Win,Cricsheet,66283.0
1,2002-06-29,England,India,"Lord's, London",London,England,bat,India,Win,Cricsheet,66284.0
2,2002-12-29,New Zealand,India,"McLean Park, Napier",Napier,India,field,New Zealand,Win,Cricsheet,64814.0
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",Christchurch,India,bat,New Zealand,Win,Cricsheet,64815.0
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",Queenstown,New Zealand,field,New Zealand,Win,Cricsheet,64816.0


In [42]:

# 1. Check missing values
print("Missing values:")
print(matches_df.isnull().sum())

# 2. Check duplicate match records
print("\nDuplicate rows:", matches_df.duplicated().sum())

# 3. Check missing team names
print("\nMissing team names:")
print("Team 1:", matches_df["team_1"].isnull().sum())
print("Team 2:", matches_df["team_2"].isnull().sum())

# 4. Check matches where both teams are identical
same_team_matches = matches_df[
    matches_df["team_1"] == matches_df["team_2"]
]

print("\nMatches with identical team names:", len(same_team_matches))

# 5. Check result categories
print("\nUnique result categories:")
print(matches_df["result_type"].unique())

# 6. Check whether winners belong to either participating team
valid_winner = (
    matches_df["winner"].isna()
    | (matches_df["winner"] == matches_df["team_1"])
    | (matches_df["winner"] == matches_df["team_2"])
)

print("\nInvalid winner records:", (~valid_winner).sum())


Missing values:
match_date         0
team_1             0
team_2             0
venue              0
city               0
toss_winner        4
toss_decision      4
winner           139
result_type        0
source             0
match_id         153
dtype: int64

Duplicate rows: 0

Missing team names:
Team 1: 0
Team 2: 0

Matches with identical team names: 0

Unique result categories:
['Win' 'No result' 'Tie']

Invalid winner records: 0


In [43]:

# Create a copy for feature engineering
features_df = matches_df.copy()

# Create target variable
features_df["team_1_win"] = np.where(
    features_df["winner"] == features_df["team_1"],
    1,
    np.where(
        features_df["winner"] == features_df["team_2"],
        0,
        np.nan
    )
)

# Check target distribution
print("Target distribution:")
print(features_df["team_1_win"].value_counts(dropna=False))

print("\nTarget percentage:")
print(
    features_df["team_1_win"]
    .value_counts(normalize=True, dropna=False) * 100
)

print("\nSample records:")
display(
    features_df[
        ["match_date", "team_1", "team_2", "winner",
         "result_type", "team_1_win"]
    ].head(10)
)


Target distribution:
team_1_win
1.0    1418
0.0    1196
NaN     139
Name: count, dtype: int64

Target percentage:
team_1_win
1.0    51.507446
0.0    43.443516
NaN     5.049037
Name: proportion, dtype: float64

Sample records:


,match_date,team_1,team_2,winner,result_type,team_1_win
0,2002-06-27,England,Sri Lanka,England,Win,1.0
1,2002-06-29,England,India,India,Win,0.0
2,2002-12-29,New Zealand,India,New Zealand,Win,1.0
3,2003-01-01,India,New Zealand,New Zealand,Win,0.0
4,2003-01-04,India,New Zealand,New Zealand,Win,0.0
5,2003-01-08,New Zealand,India,India,Win,0.0
6,2003-01-09,Sri Lanka,Australia,Sri Lanka,Win,1.0
7,2003-01-11,Australia,England,Australia,Win,1.0
8,2003-01-13,Sri Lanka,England,Sri Lanka,Win,1.0
9,2003-01-14,India,New Zealand,New Zealand,Win,0.0


In [44]:

# Sort matches chronologically
features_df = features_df.sort_values(
    by=["match_date"]
).reset_index(drop=True)

# Check chronological ordering
print("Dataset sorted chronologically:",
      features_df["match_date"].is_monotonic_increasing)

# Count matches per date
matches_per_date = features_df.groupby("match_date").size()

print("\nNumber of unique match dates:", len(matches_per_date))
print("Dates with multiple matches:", (matches_per_date > 1).sum())
print("Maximum matches on a single date:", matches_per_date.max())
print()
# Display dates with multiple matches
print("\nSample dates with multiple matches:")
display(matches_per_date[matches_per_date > 1].head(10))

# Display first and last 5 records
print("\nEarliest matches:")
display(features_df.head())

print("\nLatest matches:")
display(features_df.tail())


Dataset sorted chronologically: True

Number of unique match dates: 2326
Dates with multiple matches: 396
Maximum matches on a single date: 4


Sample dates with multiple matches:


match_date
2003-02-10    2
2003-02-11    2
2003-02-12    2
2003-02-15    2
2003-02-16    3
2003-02-19    3
2003-02-22    2
2003-02-23    2
2003-02-24    2
2003-02-26    2
dtype: int64


Earliest matches:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id,team_1_win
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",Nottingham,England,bat,England,Win,Cricsheet,66283.0,1.0
1,2002-06-29,England,India,"Lord's, London",London,England,bat,India,Win,Cricsheet,66284.0,0.0
2,2002-12-29,New Zealand,India,"McLean Park, Napier",Napier,India,field,New Zealand,Win,Cricsheet,64814.0,1.0
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",Christchurch,India,bat,New Zealand,Win,Cricsheet,64815.0,0.0
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",Queenstown,New Zealand,field,New Zealand,Win,Cricsheet,64816.0,0.0



Latest matches:


,match_date,team_1,team_2,venue,city,toss_winner,toss_decision,winner,result_type,source,match_id,team_1_win
2748,2026-10-22,Ireland,Namibia,"High Performance Oval, Windhoek",Windhoek,Namibia,field,Ireland,Win,Cricsheet,1558231.0,1.0
2749,2026-10-23,Afghanistan,Zimbabwe,Sharjah Cricket Stadium,Sharjah,Zimbabwe,bat,Afghanistan,Win,Crickrida,NaN,1.0
2750,2026-10-24,Namibia,Ireland,"High Performance Oval, Windhoek",Windhoek,Ireland,field,Namibia,Win,Cricsheet,1558232.0,1.0
2751,2026-10-25,England,Pakistan,"Gaddafi Stadium, Lahore",Lahore,Pakistan,field,Pakistan,Win,Cricsheet,1550716.0,0.0
2752,2026-10-25,United Arab Emirates,Netherlands,"ICC Academy, Dubai",Dubai,United Arab Emirates,bat,Netherlands,Win,Cricsheet,1559563.0,0.0


In [45]:

# Store historical wins and decisive matches for each team
team_wins = {}
team_matches = {}

# Create columns for historical performance
features_df["team_1_win_pct"] = np.nan
features_df["team_2_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate historical win percentages BEFORE today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        team1_wins = team_wins.get(team1, 0)
        team1_matches = team_matches.get(team1, 0)

        team2_wins = team_wins.get(team2, 0)
        team2_matches = team_matches.get(team2, 0)

        # Use 0.5 as the neutral starting estimate for teams
        features_df.at[idx, "team_1_win_pct"] = (
            team1_wins / team1_matches
            if team1_matches > 0 else 0.5
        )

        features_df.at[idx, "team_2_win_pct"] = (
            team2_wins / team2_matches
            if team2_matches > 0 else 0.5
        )

    # Update historical statistics AFTER processing all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        # Update statistics only for decisive matches
        if pd.notna(winner):

            team_matches[team1] = team_matches.get(team1, 0) + 1
            team_matches[team2] = team_matches.get(team2, 0) + 1

            if winner == team1:
                team_wins[team1] = team_wins.get(team1, 0) + 1

            elif winner == team2:
                team_wins[team2] = team_wins.get(team2, 0) + 1

# Calculate the difference in historical win percentage
features_df["win_pct_difference"] = (
    features_df["team_1_win_pct"] -
    features_df["team_2_win_pct"]
)

# Inspect results
print("Historical win percentage features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_win_pct", "team_2_win_pct",
         "win_pct_difference", "winner"]
    ].head(15)
)


Historical win percentage features created!


,match_date,team_1,team_2,team_1_win_pct,team_2_win_pct,win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,0.500000,0.500000,0.000000,England
1,2002-06-29,England,India,1.000000,0.500000,0.500000,India
2,2002-12-29,New Zealand,India,0.500000,1.000000,-0.500000,New Zealand
3,2003-01-01,India,New Zealand,0.500000,1.000000,-0.500000,New Zealand
4,2003-01-04,India,New Zealand,0.333333,1.000000,-0.666667,New Zealand
5,2003-01-08,New Zealand,India,1.000000,0.250000,0.750000,India
6,2003-01-09,Sri Lanka,Australia,0.000000,0.500000,-0.500000,Sri Lanka
7,2003-01-11,Australia,England,0.000000,0.500000,-0.500000,Australia
8,2003-01-13,Sri Lanka,England,0.500000,0.333333,0.166667,Sri Lanka
9,2003-01-14,India,New Zealand,0.400000,0.750000,-0.350000,New Zealand


In [46]:

# Store the last 5 decisive results for each team
team_recent_results = {}

# Create recent form columns
features_df["team_1_recent_form"] = np.nan
features_df["team_2_recent_form"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate recent form BEFORE today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        team1_history = team_recent_results.get(team1, [])
        team2_history = team_recent_results.get(team2, [])

        # Neutral 50% if fewer than 5 previous decisive matches exist
        features_df.at[idx, "team_1_recent_form"] = (
            np.mean(team1_history[-5:])
            if team1_history else 0.5
        )

        features_df.at[idx, "team_2_recent_form"] = (
            np.mean(team2_history[-5:])
            if team2_history else 0.5
        )

    # Update recent results only after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):

            team1_result = 1 if winner == team1 else 0
            team2_result = 1 if winner == team2 else 0

            team_recent_results.setdefault(team1, []).append(team1_result)
            team_recent_results.setdefault(team2, []).append(team2_result)

# Calculate recent form difference
features_df["recent_form_difference"] = (
    features_df["team_1_recent_form"] -
    features_df["team_2_recent_form"]
)

# Display sample results
print("Recent form features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_recent_form", "team_2_recent_form",
         "recent_form_difference", "winner"]
    ].head(20)
)
display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_recent_form", "team_2_recent_form",
         "recent_form_difference", "winner"]
    ].tail(20)
)


Recent form features created!


,match_date,team_1,team_2,team_1_recent_form,team_2_recent_form,recent_form_difference,winner
0,2002-06-27,England,Sri Lanka,0.500000,0.500000,0.000000,England
1,2002-06-29,England,India,1.000000,0.500000,0.500000,India
2,2002-12-29,New Zealand,India,0.500000,1.000000,-0.500000,New Zealand
3,2003-01-01,India,New Zealand,0.500000,1.000000,-0.500000,New Zealand
4,2003-01-04,India,New Zealand,0.333333,1.000000,-0.666667,New Zealand
5,2003-01-08,New Zealand,India,1.000000,0.250000,0.750000,India
6,2003-01-09,Sri Lanka,Australia,0.000000,0.500000,-0.500000,Sri Lanka
7,2003-01-11,Australia,England,0.000000,0.500000,-0.500000,Australia
8,2003-01-13,Sri Lanka,England,0.500000,0.333333,0.166667,Sri Lanka
9,2003-01-14,India,New Zealand,0.400000,0.750000,-0.350000,New Zealand


,match_date,team_1,team_2,team_1_recent_form,team_2_recent_form,recent_form_difference,winner
2733,2026-10-09,Oman,Canada,0.2,0.2,0.0,Oman
2734,2026-10-09,United Arab Emirates,Namibia,0.4,0.2,0.2,Namibia
2735,2026-10-11,Nepal,Canada,0.4,0.0,0.4,Nepal
2736,2026-10-11,United Arab Emirates,United States of America,0.4,0.8,-0.4,United States of America
2737,2026-10-13,United States of America,Namibia,0.8,0.2,0.6,Namibia
2738,2026-10-13,Nepal,Oman,0.6,0.2,0.4,Nepal
2739,2026-10-15,Canada,Oman,0.0,0.2,-0.2,Canada
2740,2026-10-17,Canada,Nepal,0.2,0.8,-0.6,Canada
2741,2026-10-17,Zimbabwe,Bangladesh,0.2,0.4,-0.2,Zimbabwe
2742,2026-10-18,Pakistan,Sri Lanka,0.6,0.4,0.2,Pakistan


In [47]:

# Store previous head-to-head results
head_to_head = {}

# Create head-to-head feature columns
features_df["team_1_h2h_win_pct"] = np.nan
features_df["team_2_h2h_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate head-to-head performance before today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]

        pair = tuple(sorted([team1, team2]))
        previous_results = head_to_head.get(pair, [])

        if previous_results:
            team1_wins = sum(
                1 for winner in previous_results if winner == team1
            )
            team1_pct = team1_wins / len(previous_results)
            team2_pct = 1 - team1_pct
        else:
            team1_pct = 0.5
            team2_pct = 0.5

        features_df.at[idx, "team_1_h2h_win_pct"] = team1_pct
        features_df.at[idx, "team_2_h2h_win_pct"] = team2_pct

    # Update head-to-head history after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):
            pair = tuple(sorted([team1, team2]))
            head_to_head.setdefault(pair, []).append(winner)

# Calculate head-to-head difference
features_df["h2h_win_pct_difference"] = (
    features_df["team_1_h2h_win_pct"] -
    features_df["team_2_h2h_win_pct"]
)

print("Head-to-head features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2",
         "team_1_h2h_win_pct", "team_2_h2h_win_pct",
         "h2h_win_pct_difference", "winner"]
    ].head(20)
)


Head-to-head features created!


,match_date,team_1,team_2,team_1_h2h_win_pct,team_2_h2h_win_pct,h2h_win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,0.50,0.50,0.0,England
1,2002-06-29,England,India,0.50,0.50,0.0,India
2,2002-12-29,New Zealand,India,0.50,0.50,0.0,New Zealand
3,2003-01-01,India,New Zealand,0.00,1.00,-1.0,New Zealand
4,2003-01-04,India,New Zealand,0.00,1.00,-1.0,New Zealand
5,2003-01-08,New Zealand,India,1.00,0.00,1.0,India
6,2003-01-09,Sri Lanka,Australia,0.50,0.50,0.0,Sri Lanka
7,2003-01-11,Australia,England,0.50,0.50,0.0,Australia
8,2003-01-13,Sri Lanka,England,0.00,1.00,-1.0,Sri Lanka
9,2003-01-14,India,New Zealand,0.25,0.75,-0.5,New Zealand


In [48]:

# Store previous team results at each venue
venue_history = {}

# Create venue feature columns
features_df["team_1_venue_win_pct"] = np.nan
features_df["team_2_venue_win_pct"] = np.nan

# Process matches date by date
for match_date, daily_matches in features_df.groupby("match_date", sort=True):

    # Calculate venue performance before today's matches
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        venue = features_df.at[idx, "venue"]

        team1_history = venue_history.get((team1, venue), [])
        team2_history = venue_history.get((team2, venue), [])

        features_df.at[idx, "team_1_venue_win_pct"] = (
            np.mean(team1_history) if team1_history else 0.5
        )

        features_df.at[idx, "team_2_venue_win_pct"] = (
            np.mean(team2_history) if team2_history else 0.5
        )

    # Update venue history after all matches on this date
    for idx in daily_matches.index:

        team1 = features_df.at[idx, "team_1"]
        team2 = features_df.at[idx, "team_2"]
        venue = features_df.at[idx, "venue"]
        winner = features_df.at[idx, "winner"]

        if pd.notna(winner):

            team1_result = 1 if winner == team1 else 0
            team2_result = 1 if winner == team2 else 0

            venue_history.setdefault((team1, venue), []).append(team1_result)
            venue_history.setdefault((team2, venue), []).append(team2_result)

# Calculate venue performance difference
features_df["venue_win_pct_difference"] = (
    features_df["team_1_venue_win_pct"] -
    features_df["team_2_venue_win_pct"]
)

print("Venue performance features created!")

display(
    features_df[
        ["match_date", "team_1", "team_2", "venue",
         "team_1_venue_win_pct", "team_2_venue_win_pct",
         "venue_win_pct_difference", "winner"]
    ].head(20)
)


Venue performance features created!


,match_date,team_1,team_2,venue,team_1_venue_win_pct,team_2_venue_win_pct,venue_win_pct_difference,winner
0,2002-06-27,England,Sri Lanka,"Trent Bridge, Nottingham",0.5,0.5,0.0,England
1,2002-06-29,England,India,"Lord's, London",0.5,0.5,0.0,India
2,2002-12-29,New Zealand,India,"McLean Park, Napier",0.5,0.5,0.0,New Zealand
3,2003-01-01,India,New Zealand,"Jade Stadium, Christchurch",0.5,0.5,0.0,New Zealand
4,2003-01-04,India,New Zealand,"Davies Park, Queenstown",0.5,0.5,0.0,New Zealand
5,2003-01-08,New Zealand,India,"Westpac Stadium, Wellington",0.5,0.5,0.0,India
6,2003-01-09,Sri Lanka,Australia,Sydney Cricket Ground,0.5,0.5,0.0,Sri Lanka
7,2003-01-11,Australia,England,"Bellerive Oval, Hobart",0.5,0.5,0.0,Australia
8,2003-01-13,Sri Lanka,England,Sydney Cricket Ground,1.0,0.5,0.5,Sri Lanka
9,2003-01-14,India,New Zealand,"Westpac Park, Hamilton",0.5,0.5,0.0,New Zealand


In [49]:
# Identify ties and no-result matches
tie_matches = features_df[features_df["result_type"] == "Tie"]
no_result_matches = features_df[features_df["result_type"] == "No result"]

print("Tie matches:", len(tie_matches))
print("No-result matches:", len(no_result_matches))

print("\nTie match Elo samples:")
display(
    tie_matches[
        ["match_date", "team_1", "team_2",
         "team_1_elo", "team_2_elo", "elo_difference"]
    ].head()
)

print("\nNo-result match Elo samples:")
display(
    no_result_matches[
        ["match_date", "team_1", "team_2",
         "team_1_elo", "team_2_elo", "elo_difference"]
    ].head()
)

Tie matches: 28
No-result matches: 111

Tie match Elo samples:


KeyError: "['team_1_elo', 'team_2_elo', 'elo_difference'] not in index"

In [ ]:
tie_actual_score = 0.5
no_result_update = 0

print("Expected Elo behaviour:")
print("Tie: Both teams receive an actual score of 0.5")
print("No result: Neither team's rating changes")

# Check the Elo update formula for a tied match
sample_tie = tie_matches.iloc[0]

team_1_expected = calculate_expected_score(
    sample_tie["team_1_elo"],
    sample_tie["team_2_elo"]
)

team_2_expected = calculate_expected_score(
    sample_tie["team_2_elo"],
    sample_tie["team_1_elo"]
)

team_1_change = K_FACTOR * (tie_actual_score - team_1_expected)
team_2_change = K_FACTOR * (tie_actual_score - team_2_expected)

print("\nSample tie:", sample_tie["team_1"], "vs", sample_tie["team_2"])
print("Team 1 expected score:", team_1_expected)
print("Team 2 expected score:", team_2_expected)
print("Team 1 Elo change:", team_1_change)
print("Team 2 Elo change:", team_2_change)
print("Total Elo change:", team_1_change + team_2_change)

Expected Elo behaviour:
Tie: Both teams receive an actual score of 0.5
No result: Neither team's rating changes

Sample tie: Sri Lanka vs South Africa
Team 1 expected score: 0.48311024961816873
Team 2 expected score: 0.5168897503818313
Team 1 Elo change: 0.5404720122186006
Team 2 Elo change: -0.5404720122186006
Total Elo change: 0.0


In [ ]:
# Check that no-result matches are skipped during Elo updates

print("No-result handling in Elo calculation:")

print("""
if match["result_type"] == "No result":
    continue
""")

print("No-result matches:", len(no_result_matches))
print("Expected Elo update for no-result matches: 0")

No-result handling in Elo calculation:

if match["result_type"] == "No result":
    continue

No-result matches: 111
Expected Elo update for no-result matches: 0


In [ ]:
# List all columns currently available
print("Dataset shape:", features_df.shape)

print("\nAll columns:")
for i, col in enumerate(features_df.columns, start=1):
    print(f"{i}. {col}")

print("\nMissing values in engineered features:")
engineered_cols = [
    col for col in features_df.columns
    if col not in matches_df.columns
]

print(features_df[engineered_cols].isnull().sum())

print("\nNumber of engineered features:", len(engineered_cols))

Dataset shape: (2753, 40)

All columns:
1. match_date
2. team_1
3. team_2
4. venue
5. city
6. toss_winner
7. toss_decision
8. winner
9. result_type
10. source
11. match_id
12. team_1_win
13. team_1_win_pct
14. team_2_win_pct
15. win_pct_difference
16. team_1_recent_form
17. team_2_recent_form
18. recent_form_difference
19. team_1_h2h_win_pct
20. team_2_h2h_win_pct
21. h2h_win_pct_difference
22. team_1_venue_win_pct
23. team_2_venue_win_pct
24. venue_win_pct_difference
25. city_original
26. city_standard
27. venue_country
28. team_1_is_home
29. team_2_is_home
30. neutral_venue
31. home_advantage
32. team_1_home_win_pct
33. team_2_home_win_pct
34. team_1_away_win_pct
35. team_2_away_win_pct
36. home_win_pct_difference
37. away_win_pct_difference
38. team_1_elo
39. team_2_elo
40. elo_difference

Missing values in engineered features:
team_1_win                  139
team_1_win_pct                0
team_2_win_pct                0
win_pct_difference            0
team_1_recent_form           

In [ ]:
# Validate percentage-based features

percentage_cols = [
    "team_1_win_pct",
    "team_2_win_pct",
    "team_1_recent_form",
    "team_2_recent_form",
    "team_1_h2h_win_pct",
    "team_2_h2h_win_pct",
    "team_1_venue_win_pct",
    "team_2_venue_win_pct",
    "team_1_home_win_pct",
    "team_2_home_win_pct",
    "team_1_away_win_pct",
    "team_2_away_win_pct"
]

print("Percentage features within 0–1:")
print(
    features_df[percentage_cols]
    .apply(lambda col: col.between(0, 1).all())
)

# Validate binary features
binary_cols = [
    "team_1_is_home",
    "team_2_is_home",
    "neutral_venue",
    "team_1_win"
]

print("\nBinary feature values:")
for col in binary_cols:
    print(col, sorted(features_df[col].dropna().unique()))

# Validate home advantage
print("\nHome advantage values:")
print(sorted(features_df["home_advantage"].unique()))

Percentage features within 0–1:
team_1_win_pct          True
team_2_win_pct          True
team_1_recent_form      True
team_2_recent_form      True
team_1_h2h_win_pct      True
team_2_h2h_win_pct      True
team_1_venue_win_pct    True
team_2_venue_win_pct    True
team_1_home_win_pct     True
team_2_home_win_pct     True
team_1_away_win_pct     True
team_2_away_win_pct     True
dtype: bool

Binary feature values:
team_1_is_home [np.int64(0), np.int64(1)]
team_2_is_home [np.int64(0), np.int64(1)]
neutral_venue [np.int64(0), np.int64(1)]
team_1_win [np.float64(0.0), np.float64(1.0)]

Home advantage values:
[np.int64(-1), np.int64(0), np.int64(1)]


In [ ]:
difference_checks = {
    "win_pct_difference": (
        "team_1_win_pct", "team_2_win_pct"
    ),
    "recent_form_difference": (
        "team_1_recent_form", "team_2_recent_form"
    ),
    "h2h_win_pct_difference": (
        "team_1_h2h_win_pct", "team_2_h2h_win_pct"
    ),
    "venue_win_pct_difference": (
        "team_1_venue_win_pct", "team_2_venue_win_pct"
    ),
    "home_win_pct_difference": (
        "team_1_home_win_pct", "team_2_home_win_pct"
    ),
    "away_win_pct_difference": (
        "team_1_away_win_pct", "team_2_away_win_pct"
    ),
    "elo_difference": (
        "team_1_elo", "team_2_elo"
    )
}

for difference_col, (team1_col, team2_col) in difference_checks.items():
    is_valid = np.allclose(
        features_df[difference_col],
        features_df[team1_col] - features_df[team2_col]
    )
    print(f"{difference_col}: {is_valid}")

win_pct_difference: True
recent_form_difference: True
h2h_win_pct_difference: True
venue_win_pct_difference: True
home_win_pct_difference: True
away_win_pct_difference: True
elo_difference: True


In [ ]:
# Save the engineered dataset

FEATURE_FILE = FEATURE_DIR / "cricketiq_engineered_features.csv"

features_df.to_csv(FEATURE_FILE, index=False)

print("Feature dataset saved successfully!")
print("File:", FEATURE_FILE)
print("Shape:", features_df.shape)
print("Total matches:", len(features_df))
print("Total columns:", len(features_df.columns))

Feature dataset saved successfully!
File: C:\CricIQ\data\features\cricketiq_engineered_features.csv
Shape: (2753, 40)
Total matches: 2753
Total columns: 40


In [ ]:
# Reload the saved feature dataset
saved_features_df = pd.read_csv(FEATURE_FILE)

print("Reloaded dataset shape:", saved_features_df.shape)

print("\nColumn count:", len(saved_features_df.columns))

print("\nMissing values:")
print(saved_features_df.isnull().sum().sum())

print("\nSource distribution:")
print(saved_features_df["source"].value_counts())

print("\nResult distribution:")
print(saved_features_df["result_type"].value_counts())

# Verify original and reloaded data
print("\nShape preserved:", saved_features_df.shape == features_df.shape)
print("Column names preserved:", list(saved_features_df.columns) == list(features_df.columns))
print(
    "Match dates preserved:",
    pd.to_datetime(saved_features_df["match_date"]).equals(
        pd.to_datetime(features_df["match_date"])
    )
)

Reloaded dataset shape: (2753, 40)

Column count: 40

Missing values:
439

Source distribution:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

Result distribution:
result_type
Win          2614
No result     111
Tie            28
Name: count, dtype: int64

Shape preserved: True
Column names preserved: True
Match dates preserved: True


In [ ]:
# Identify missing values in the reloaded dataset

missing_summary = (
    saved_features_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("Columns with missing values:")
print(missing_summary[missing_summary > 0])

print("\nTotal missing values:", saved_features_df.isnull().sum().sum())

print("\nOriginal in-memory missing values:")
print(features_df.isnull().sum().sum())

Columns with missing values:
match_id         153
winner           139
team_1_win       139
toss_winner        4
toss_decision      4
dtype: int64

Total missing values: 439

Original in-memory missing values:
439


In [1]:

import json
from pathlib import Path

path = Path(r"C:\CricIQ\notebooks\05_feature_engineering.ipynb")

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

cells = nb["cells"]

print("=" * 75)
print("CRICKETIQ — NOTEBOOK 05 FEATURE ENGINEERING AUDIT")
print("=" * 75)

print("Total cells:", len(cells))
print("Code cells:", sum(c["cell_type"] == "code" for c in cells))
print("Markdown cells:", sum(c["cell_type"] == "markdown" for c in cells))

print("\n" + "=" * 75)
print("1. DOCUMENTATION HEADINGS")
print("=" * 75)

for i, cell in enumerate(cells):
    if cell["cell_type"] == "markdown":
        source = "".join(cell.get("source", [])).strip()
        if source:
            print(f"\nCell {i}:")
            print(source)

print("\n" + "=" * 75)
print("2. CODE CELL INVENTORY")
print("=" * 75)

for i, cell in enumerate(cells):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", [])).strip()
    first_line = source.splitlines()[0] if source else "[EMPTY]"

    print(f"Cell {i:>3}: {first_line[:120]}")

print("\n" + "=" * 75)
print("3. ELO AND FEATURE ENGINEERING CODE")
print("=" * 75)

keywords = [
    "elo", "rating", "feature", "engineer",
    "rolling", "momentum", "win_rate",
    "home_advantage", "target"
]

for i, cell in enumerate(cells):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(word in source.lower() for word in keywords):
        print(f"\n--- CELL {i} ---")
        print(source[:5000])

print("\nINSPECTION COMPLETE — NO MODIFICATIONS")


CRICKETIQ — NOTEBOOK 05 FEATURE ENGINEERING AUDIT
Total cells: 58
Code cells: 58
Markdown cells: 0

1. DOCUMENTATION HEADINGS

2. CODE CELL INVENTORY
Cell   0: import sys
Cell   1: import pandas as pd
Cell   2: # 1. Check missing values
Cell   3: # Create a copy for feature engineering
Cell   4: # Sort matches chronologically
Cell   5: # Store historical wins and decisive matches for each team
Cell   6: # Store the last 5 decisive results for each team
Cell   7: # Store previous head-to-head results
Cell   8: # Store previous team results at each venue
Cell   9: # Inspect unique venues and cities
Cell  10: # Display all unique teams
Cell  11: # Map each team to its home country
Cell  12: # Divide unique cities into groups of 40 for easier mapping
Cell  13: # Standardise known city-name variations
Cell  14: # City-to-country mapping: Group 1
Cell  15: group_1_standard = [
Cell  16: # City-to-country mapping: Group 2
Cell  17: # City-to-country mapping: Group 3
Cell  18: # City-to-countr

In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

feature_path = Path(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

if not feature_path.exists():
    print("FEATURE DATASET NOT FOUND:", feature_path)

else:
    df = pd.read_csv(feature_path)

    print("=" * 75)
    print("CRICKETIQ — SAVED FEATURE DATASET AUDIT")
    print("=" * 75)

    print("\nShape:", df.shape)
    print("Total columns:", len(df.columns))

    print("\nColumns:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:02}. {col}")

    print("\nSource distribution:")
    print(df["source"].value_counts(dropna=False))

    print("\nResult distribution:")
    print(df["result_type"].value_counts(dropna=False))

    print("\nELO AUDIT")
    elo_cols = [
        "team_1_elo",
        "team_2_elo",
        "elo_difference"
    ]

    for col in elo_cols:
        if col in df.columns:
            print(
                f"{col}: "
                f"missing={df[col].isna().sum()}, "
                f"min={df[col].min()}, "
                f"max={df[col].max()}"
            )
        else:
            print(f"{col}: MISSING")

    print("\nENGINEERED FEATURE MISSING VALUES")
    base_cols = [
        "match_date", "team_1", "team_2", "venue",
        "city", "toss_winner", "toss_decision",
        "winner", "result_type", "source", "match_id"
    ]

    engineered_cols = [
        col for col in df.columns if col not in base_cols
    ]

    print("Engineered columns:", len(engineered_cols))
    print(df[engineered_cols].isna().sum().to_string())

    print("\nDUPLICATE MATCH CHECK")
    print("Duplicate complete rows:", df.duplicated().sum())

    print("\nAUDIT COMPLETE — NO FILES MODIFIED")


CRICKETIQ — SAVED FEATURE DATASET AUDIT

Shape: (2753, 40)
Total columns: 40

Columns:
01. match_date
02. team_1
03. team_2
04. venue
05. city
06. toss_winner
07. toss_decision
08. winner
09. result_type
10. source
11. match_id
12. team_1_win
13. team_1_win_pct
14. team_2_win_pct
15. win_pct_difference
16. team_1_recent_form
17. team_2_recent_form
18. recent_form_difference
19. team_1_h2h_win_pct
20. team_2_h2h_win_pct
21. h2h_win_pct_difference
22. team_1_venue_win_pct
23. team_2_venue_win_pct
24. venue_win_pct_difference
25. city_original
26. city_standard
27. venue_country
28. team_1_is_home
29. team_2_is_home
30. neutral_venue
31. home_advantage
32. team_1_home_win_pct
33. team_2_home_win_pct
34. team_1_away_win_pct
35. team_2_away_win_pct
36. home_win_pct_difference
37. away_win_pct_difference
38. team_1_elo
39. team_2_elo
40. elo_difference

Source distribution:
source
Cricsheet    2600
Crickrida     153
Name: count, dtype: int64

Result distribution:
result_type
Win          261

In [3]:

import pandas as pd

df = pd.read_csv(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

# Standardize temporary comparison fields
audit = df.copy()

audit["match_date"] = pd.to_datetime(
    audit["match_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

audit["team_pair"] = audit.apply(
    lambda row: " | ".join(
        sorted([str(row["team_1"]).strip(),
                str(row["team_2"]).strip()])
    ),
    axis=1
)

audit["fixture_key"] = (
    audit["match_date"].astype(str)
    + " | " + audit["team_pair"]
)

print("=" * 70)
print("CROSS-SOURCE MATCH OVERLAP AUDIT")
print("=" * 70)

# 1. Match ID uniqueness
print("\nDuplicate match IDs:")
print(audit["match_id"].duplicated().sum())

# 2. Duplicate fixtures within each source
print("\nDuplicate fixture keys within each source:")
print(
    audit.groupby("source")["fixture_key"]
    .apply(lambda s: s.duplicated().sum())
)

# 3. Fixtures appearing in both sources
source_counts = audit.groupby("fixture_key")["source"].nunique()

overlap_keys = source_counts[source_counts > 1].index

overlap = audit[
    audit["fixture_key"].isin(overlap_keys)
].sort_values(["fixture_key", "source"])

print("\nFixtures appearing in both sources:", len(overlap_keys))

if len(overlap):
    display(
        overlap[
            ["match_date", "team_1", "team_2",
             "source", "match_id", "winner", "result_type"]
        ].head(30)
    )
else:
    print("No cross-source fixture overlaps found.")

print("\nAUDIT COMPLETE — ORIGINAL DATA NOT MODIFIED")


CROSS-SOURCE MATCH OVERLAP AUDIT

Duplicate match IDs:
152

Duplicate fixture keys within each source:
source
Crickrida    0
Cricsheet    0
Name: fixture_key, dtype: int64

Fixtures appearing in both sources: 0
No cross-source fixture overlaps found.

AUDIT COMPLETE — ORIGINAL DATA NOT MODIFIED


In [4]:

import pandas as pd

df = pd.read_csv(
    r"C:\CricIQ\data\features\cricketiq_engineered_features.csv"
)

duplicate_ids = (
    df[df["match_id"].duplicated(keep=False)]
    .sort_values(["match_id", "source", "match_date"])
)

print("=" * 70)
print("DUPLICATE MATCH ID INVESTIGATION")
print("=" * 70)

print("\nTotal rows involved:", len(duplicate_ids))
print("Unique duplicated IDs:", duplicate_ids["match_id"].nunique())

print("\nDuplicate IDs by source:")
print(
    duplicate_ids.groupby("source")["match_id"]
    .nunique()
)

print("\nSource combinations for duplicated IDs:")
print(
    duplicate_ids.groupby("match_id")["source"]
    .agg(lambda x: ", ".join(sorted(set(x))))
    .value_counts()
)

print("\nFirst 40 duplicate-ID records:")
display(
    duplicate_ids[
        ["match_id", "match_date", "team_1", "team_2",
         "venue", "source", "winner", "result_type"]
    ].head(40)
)

print("\nAUDIT COMPLETE — NO FILES MODIFIED")


DUPLICATE MATCH ID INVESTIGATION

Total rows involved: 153
Unique duplicated IDs: 0

Duplicate IDs by source:
source
Crickrida    0
Name: match_id, dtype: int64

Source combinations for duplicated IDs:
Series([], Name: count, dtype: int64)

First 40 duplicate-ID records:


,match_id,match_date,team_1,team_2,venue,source,winner,result_type
807,NaN,2010-07-03,Afghanistan,Ireland,Hazelaarweg,Crickrida,Ireland,Win
1000,NaN,2012-02-10,Afghanistan,Pakistan,Sharjah Cricket Stadium,Crickrida,Pakistan,Win
1044,NaN,2012-07-03,Afghanistan,Ireland,Clontarf Cricket Club Ground,Crickrida,NaN,No result
1046,NaN,2012-07-05,Afghanistan,Ireland,Clontarf Cricket Club Ground,Crickrida,Ireland,Win
1059,NaN,2012-08-25,Afghanistan,Australia,Sharjah Cricket Stadium,Crickrida,Australia,Win
1224,NaN,2014-02-27,Afghanistan,Pakistan,Khan Shaheb Osman Ali Stadium,Crickrida,Pakistan,Win
1227,NaN,2014-03-01,Afghanistan,Bangladesh,Khan Shaheb Osman Ali Stadium,Crickrida,Afghanistan,Win
1230,NaN,2014-03-03,Afghanistan,Sri Lanka,Shere Bangla National Stadium,Crickrida,Sri Lanka,Win
1233,NaN,2014-03-05,Afghanistan,India,Shere Bangla National Stadium,Crickrida,India,Win
1250,NaN,2014-07-18,Afghanistan,Zimbabwe,Queens Sports Club,Crickrida,Zimbabwe,Win



AUDIT COMPLETE — NO FILES MODIFIED
